In [ ]:
import time
from datetime import datetime

START = "start"
END = "end"
PROG_TIME = "Program Time"
IMPORTS = "Imports"
SYS_INFO = "Sys Info"
UTIL_FUNCS = "Utility functions"
DATA_LOAD = "Loading data"
EDA = "EDA"
EXPERIMENTS = "Experiments"
SCALING = "Scaling"
SPLIT = "Train-Test Split"
ANN_TUNING = "Tuning ANN"
XGB_TUNING = "Tuning XGB"
RF_TUNING = "Tuning Random Forest"
MODEL_DEPLOYMENT = "Model Deployment"

timings = {} # dictionary to hold the start and end times of various sections and the entire program itself
sec_timing = {START: 0.00, END: 0.00}

def record_time(time_reg, time_key, time):
    if(time_reg is None):
        time_reg = sec_timing.copy()
    time_reg[time_key] = time
    return time_reg

def format_time(time_secs, format = "%Y-%m-%d %H:%M:%S"):
    dt = datetime.fromtimestamp(time_secs)
    return dt.strftime(format)

# track the total time from start to end of the program
prog_time = record_time(None, START, time.time())
timings[PROG_TIME] = prog_time

In [ ]:
# track the total time for section
sec_time = record_time(None, START, time.time())
timings[IMPORTS] = sec_time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import zscore
from sklearn.preprocessing import StandardScaler
import sklearn.linear_model as lm
import sklearn.ensemble as en
import sklearn.svm as svm
import sklearn.tree as tr
import sklearn.neighbors as ne
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, LeakyReLU, ReLU, Input
import keras_tuner as kt
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import AUC
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.pipeline import Pipeline
import joblib
from sklearn.metrics import roc_curve
from sklearn.metrics import precision_recall_curve, confusion_matrix
from sklearn.model_selection import ParameterSampler

import warnings
warnings.filterwarnings("ignore")

sec_time = record_time(sec_time, END, time.time())

In [ ]:
# track the total time for section
sec_time = record_time(None, START, time.time())
timings[SYS_INFO] = sec_time

# Print the system/environment info
import sys
import platform
import os
import random
import multiprocessing
import psutil

import numpy as np
import pandas as pd
import sklearn
import scipy
import joblib
import threadpoolctl
import xgboost
import lightgbm
import tensorflow as tf

print("=" * 80)
print("SYSTEM INFORMATION")
print("=" * 80)

print(f"Platform           : {platform.platform()}")
print(f"OS                 : {platform.system()} {platform.release()}")
print(f"Architecture       : {platform.architecture()[0]}")
print(f"Processor          : {platform.processor()}")
print(f"Machine            : {platform.machine()}")

# RAM info
mem = psutil.virtual_memory()

print(f"Total RAM     : {mem.total / (1024**3):.2f} GB")
print(f"Available RAM : {mem.available / (1024**3):.2f} GB")
print(f"Used RAM      : {mem.percent}%")

# current process info
process = psutil.Process(os.getpid())

print(f"Current PID      : {process.pid}")
print(f"CPU affinity     : {process.cpu_affinity()}")
print(f"Threads          : {process.num_threads()}")

# BLAS/OpenMP environment variables
for var in [
    "OMP_NUM_THREADS",
    "MKL_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "NUMEXPR_NUM_THREADS",
]:
    print(f"{var}: {os.environ.get(var)}")

print()

print("=" * 80)
print("PYTHON ENVIRONMENT")
print("=" * 80)

print(f"Python version     : {sys.version}")
print(f"Executable         : {sys.executable}")

print()

print("=" * 80)
print("LIBRARY VERSIONS")
print("=" * 80)

print(f"NumPy              : {np.__version__}")
print(f"Pandas             : {pd.__version__}")
print(f"SciPy              : {scipy.__version__}")
print(f"Scikit-Learn       : {sklearn.__version__}")
print(f"Joblib             : {joblib.__version__}")
print(f"threadpoolctl      : {threadpoolctl.__version__}")
print(f"XGBoost            : {xgboost.__version__}")
print(f"LightGBM           : {lightgbm.__version__}")
print(f"TensorFlow         : {tf.__version__}")

print()

print("=" * 80)
print("CPU INFORMATION")
print("=" * 80)

print(f"os.cpu_count()             : {os.cpu_count()}")
print(f"multiprocessing.cpu_count(): {multiprocessing.cpu_count()}")
print(f"joblib.cpu_count()         : {joblib.cpu_count()}")

print()

print("=" * 80)
print("THREADPOOL INFORMATION")
print("=" * 80)

for lib in threadpoolctl.threadpool_info():
    print(lib)

print()

print("=" * 80)
print("GPU INFORMATION")
print("=" * 80)

gpus = tf.config.list_physical_devices("GPU")

if gpus:
    print(gpus)
else:
    print("No TensorFlow GPU detected.")

print()

print("=" * 80)
print("RANDOM SEEDS")
print("=" * 80)

print("Python random module state available")
print("NumPy Random Generator available")
print("Remember to document the seeds used in your code.")

s= """This is how randomness should be made deterministic:

RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

Then at every place use:
random_state=RANDOM_STATE
"""

sec_time = record_time(sec_time, END, time.time())

### Utility Functions

In [ ]:
# track the total time for section
sec_time = record_time(None, START, time.time())
timings[UTIL_FUNCS] = sec_time
sec_time = record_time(sec_time, END, time.time())

# print given frame similar to how the default display (without print works) for better readability;
def prettyPrint(frame, maxColWidth=20, maxRows=None, leftAlignTextColumns=True):
    
    if isinstance(frame, pd.Series):
        frame = frame.to_frame()

    with pd.option_context(
            "display.width", None,
            "display.max_columns", None,
            "display.max_rows", maxRows,
            "display.expand_frame_repr", False,
            "display.max_colwidth", maxColWidth
        ):

        if(maxRows is None or len(frame) <= maxRows):
            displayFrame = frame
        else:
            ellipsis = pd.DataFrame([["..."] * len(frame.columns)], columns=frame.columns)
            displayFrame = pd.concat([frame.head(maxRows//2), ellipsis, frame.tail(maxRows//2)])

        styled = None
        if leftAlignTextColumns:
            styled = displayFrame.style.set_properties(
                subset=displayFrame.select_dtypes(include=['object', 'string']).columns,
                **{'text-align': 'left'}
            ).set_table_styles([
                {'selector': 'th', 'props': [('text-align', 'left')]}
            ])

        display(styled if styled else displayFrame)

# insert a new row after the given label in the dataframe; newRow should be a series/dictionary. For other types, behaviour is not defined
# CAUTION: this should be used only for small frames as it moves records manually
def insertRowAfterLabel(frame, newRow, label, newLabel):
    labelPos = frame.index.get_loc(label)

    # get all rows before and after label and concate in order effectively inserting newRow at desired position
    frame = pd.concat([
        frame.iloc[:labelPos+1],
        pd.DataFrame([newRow], index=[newLabel]),
        frame.iloc[labelPos+1:]
    ])

    return frame

def getBlankInfo(frame, includeNan=True):

    if isinstance(frame, pd.Series):
        frame = frame.to_frame()

    objectCols = frame.select_dtypes(include=['object', 'string'])

    if(not objectCols.empty):
        # print(f"[getBlankInfo]: objectCols: {objectCols.columns}")
        blankCountsSeries = objectCols.apply( # Compute blank (empty + whitespace) counts only for object columns
                lambda col: col.fillna("").str.strip().eq("").sum() if(includeNan) else col[col.notna()].str.strip().eq("").sum()
        )
        
        # Create full index-aligned Series initialized to 0
        blankCounts = pd.Series(0, index=frame.columns)
    
        # Assign object column results
        blankCounts[objectCols.columns] = blankCountsSeries

        return blankCounts
    else:
        print(f"[getBlankInfo]: No object/string columns present")

    return pd.Series(0, index=frame.columns)

def getBasicInfo(frame):
    """
    Get basic info about the data

    Parameters:
        frame (): the DataFrame/Series for which basic info is to be obtained

    Returns: A dictionary having the following keys:
        type: type of the frame (DataFrame/Series etc)
        rowCount: number of rows in the frame
        indexMin: minimum index value
        indexMax: maximum index value
        columnInfo: DataFrame having column name, dtype, count, null count, null percentage, blank count and blank percentage for each column
        dtypes: A Series having count of each dtype in the frame
        memoryUsageKB: memory usage in KB
    """

    blankCount = getBlankInfo(frame)

    frameType = type(frame)
    frameName = None
    if isinstance(frame, pd.Series):
        frameName = frame.name
        frame = frame.to_frame()
    
    # basicInfo = df.info() #this prints the non-null info with some frame level info only; other info requires custom handling
    frameInfo = {
        "type": frameType,
        "rowCount": len(frame),
        "frameName": frameName,
        "columnCount": len(frame.columns),
        "indexMin": frame.index.min(),
        "indexMax": frame.index.max(),
        "columnInfo": pd.DataFrame({
            'Column': frame.columns,
            'Dtype': frame.dtypes,
            'Non-Null Count': frame.notna().sum(),
            'Null Count': frame.isna().sum(),
            'Null %': round(frame.isna().sum() * 100 / len(frame), 2),
            'Blank Count': blankCount,
            'Blank %': round(blankCount * 100 / len(frame) , 2)
        }).reset_index(drop=True), #drop the older index which was "Column" column since that is added explicitly as a column
        "dtypes": frame.dtypes.value_counts(),
        "memoryUsageKB": round(frame.memory_usage(deep=False).sum() / 1024, 2)
    }
    
    return frameInfo

def printInfo(basicInfo):
    """
    Print basic info about the data in a readable format
    Parameters:
        frame (DataFrame/Series): the DataFrame/Series for which basic info is to be printed
    Returns:
        None
    """

    print(basicInfo["type"])

    # print range and column counts
    print(f"RangeIndex: {basicInfo['rowCount']} entries, {basicInfo['indexMin']} to {basicInfo['indexMax']}")

    # print("basicInfo['type'] == pd.Series:", basicInfo["type"] == pd.Series)
    # print series name if the frame is a series
    if(basicInfo["type"] == pd.Series):
        print(f"Series name: {basicInfo['frameName']}")

    print(f"Data columns (total {basicInfo['columnCount']} columns):")
    
    #print column info
    prettyPrint(basicInfo['columnInfo'], maxColWidth=50)
    
    #print counts of dtypes from basicInfo.dtypes in the format dtypes: float64(16), int64(4), string(2)
    dtype_counts = basicInfo['dtypes']
    print(f"dtypes: {', '.join(f'{dtype}({count})' for dtype, count in dtype_counts.items())}")

    #print memory usage from basicInfo.memoryUsageKB in KB
    print(f"Memory usage: {basicInfo['memoryUsageKB']} KB")

#Get records missing data - nulls/whitespaces; it can return all columns or only the problem columns for better readability
def getRecordsMissingData(frame, return_only_problem_cols=True):

    null_mask = frame.isna() # rows having nulls in any column

    str_cols = frame.select_dtypes(include=['object', 'string']) # get only object/string columns to check for whitespaces; other types would not have whitespaces and astype would convert them to string which is not desired

    if str_cols.shape[1] > 0:
        ws_mask = str_cols.astype("string").str.strip().eq("") # rows having whitespaces in any column
        ws_mask = ws_mask.reindex(columns=frame.columns, fill_value=False) # reindex to align with original frame columns and fill non-object columns with False since they can't have whitespaces
        problem_mask = null_mask | ws_mask # combine the two masks to get rows having either nulls or whitespaces in any column
    else:
        problem_mask = null_mask

    problem_row_mask = problem_mask.any(axis=1)

    if not return_only_problem_cols:
        return frame.loc[problem_row_mask] # return problem rows with all columns

    col_mask = problem_mask.loc[problem_row_mask].any(axis=0) # get columns having nulls/whitespaces only in the problem rows; this ensures that only columns relevant to the problem rows are returned
    return frame.loc[problem_row_mask, col_mask]

def replaceWhitespaces(frame, replacement):
    # old behaviour causes deprecation warning when object column after replacement only has floats;
    # it auto downcasts from object to float64 silently - this behaviour is being deprecated. To make it future proof, set the option below.
    # pd.set_option('future.no_silent_downcasting', True)
    with pd.option_context(
        "future.no_silent_downcasting", True
    ):
        return frame.replace(r"^\s*$", replacement, regex=True)

def findAndReplaceWhitespaces(frame):
    
    dfMissingData = getRecordsMissingData(frame) # get rows and columns having missing values - nulls/whitespaces
    # print("missing data before null replacement\n", dfMissingData)
    
    dfMissingData = replaceWhitespaces(dfMissingData, np.nan) # replace whitespaces with nulls for uniform handling
    # print("missing data after null replacement\n", dfMissingData)
    
    frame.loc[dfMissingData.index, dfMissingData.columns] = dfMissingData # replace values in original frame

    return dfMissingData # return specific rows/columns missing data

def convertNumericObjectsToNumeric(frame, excludeCols, debug = False):

    cols = (
        frame
        .select_dtypes(include=['object','string'])
        .columns
        .difference(excludeCols)
    )

    frame[cols] = frame[cols].apply(pd.to_numeric, errors='coerce')

    if(debug):
        print("printing types:")
        for col in cols:
            print(col, frame[col].map(type).unique())
    
        print(f"object/string cols after type conversion: {(
            frame
            .select_dtypes(include=['object','string'])
            .columns
            .difference(excludeCols)
        )}")

def changeTypes(frame, colTypes):
    for colType in colTypes:
        frame.loc[:, colType['col']] = frame.loc[:, colType['col']].astype(colType['type'])

def describeData(frame, excludeCols=None, keepNonnumericStats=False, debug=False):
    
    desc = frame.describe(include='all')

    if(not keepNonnumericStats):
        desc = desc.drop(['unique','top','freq'], errors='ignore')

    sumRow = frame.sum(numeric_only=True).reindex(frame.columns)
    desc = insertRowAfterLabel(desc, sumRow, 'count', 'sum')
    
    if(excludeCols and len(excludeCols) > 0):
        desc = desc.drop(excludeCols, axis=1, errors='ignore')
        print(f"Following columns excluded: {excludeCols}")

    if(debug):
        print(type(sumRow), sumRow)

    return desc

def printCatagoricalStats(frame: pd.DataFrame, maxRows=20, includeColumns=None, leftAlignTextColumns=True):
    if includeColumns is not None and len(includeColumns) > 0:
        catCols = includeColumns
        prettyPrint(frame[includeColumns].describe(), leftAlignTextColumns=leftAlignTextColumns)
    else:
        catCols = frame.select_dtypes(include=['object', 'string']).columns
        prettyPrint(frame.describe(include=['object', 'string']), leftAlignTextColumns=leftAlignTextColumns)

    # get the count of each categorical column
    for col in catCols:
        df = pd.DataFrame(columns=['Value', 'Count', 'Percent_Of_Total'])
        valueCounts = frame[col].value_counts()
        df["Value"] = valueCounts.index
        df["Count"] = valueCounts.values.astype(int)
        df["Percent_Of_Total"] = round(df["Count"] * 100 / len(frame), 2)

        print(f"\nValue counts for column '{col}':")
        prettyPrint(df, maxRows=maxRows, leftAlignTextColumns=leftAlignTextColumns)

def getDuplicates(frame: pd.DataFrame, cols=None):
    if cols is None or len(cols) == 0:
        cols = frame.columns.tolist()
        
    details = {'cols': cols}

    # get dupes
    dupeRowsMask = frame.duplicated(subset=cols, keep=False)
    
    # check if there are dupes
    hasDupes = dupeRowsMask.any()

    # get dupe rows
    dupeRows = frame[dupeRowsMask].sort_values(cols)

    # get the frequency of duplicated rows
    dupesFrequency = dupeRows.value_counts(subset=cols).sort_values(ascending=False)
    totalDupes = dupesFrequency.sum()

    details['hasDupes'] = hasDupes
    details['totalDupes'] = totalDupes
    details['dupesFrequency'] = dupesFrequency
    details['dupeRows'] = dupeRows

    return details

def printDupesInfo(dupeDetails, rowsToShow=20):
    if(dupeDetails['hasDupes']):
        print(f"\nTotal {dupeDetails['totalDupes']} duplicates exist for columns {dupeDetails['cols']}.")
    else:
        print(f"\nNo duplicates exist for columns {dupeDetails['cols']}.")
    
    print("\nFrequency of duplicated rows:\n", dupeDetails['dupesFrequency'])
    print("\nDuplicated rows:\n")
    prettyPrint(dupeDetails['dupeRows'], maxRows=rowsToShow)
    
    
def deduplicate(frame, cols=None, debug = False, rowsToShow=20):
    if(cols is None or len(cols) == 0):
        cols = frame.columns.tolist()

    dupeDetails = getDuplicates(frame, cols)
    if(debug):
        print("\nDuplication details before dropping full row duplicates:")
        printDupesInfo(dupeDetails, rowsToShow=rowsToShow)
    
    # drop full row duplicates only; remaining dupes can be removed after data cleanup
    frame = frame.drop_duplicates(keep='first')#.copy() # create a copy (not needed with CoW) to avoid SettingWithCOpyWarning in subsequent code. drop dupes creates a new frame but attaches a weak ref and sets _is_copy to that which triggers the warning

    if(debug):
        print("\nDuplication details after dropping full row duplicates:")
        printDupesInfo(getDuplicates(frame, cols), rowsToShow=rowsToShow)

    return frame

def nullifyMissingData(frame, debug=False):
    returnVal = {"recsMissingData_orig": None, "recsMissingData_nullified": None}
    
    # Display records missing data (nulls/whitespaces)
    recsMissingData = getRecordsMissingData(frame)
    print(f'\nOriginal rows with missing data(nulls/whitespaces):{'None' if len(recsMissingData) <= 0 else f"{len(recsMissingData)} records have missing data"}')
    
    if(len(recsMissingData) > 0):
        returnVal["recsMissingData_orig"] = recsMissingData
    
        # Replace blanks/whitespaces with null to keep missing values uniform for easier subsequent analsis/imputation etc
        dfMissingData = findAndReplaceWhitespaces(frame)
        returnVal["recsMissingData_nullified"] = dfMissingData
        
        if(debug):
            prettyPrint(recsMissingData)
            print('\nRows with missing data replaced with nulls')
            prettyPrint(frame.loc[dfMissingData.index, dfMissingData.columns]) # use frame to see if original frame was modified    

    return returnVal

def detectOutliersIQR(frame, numericCols):
    # detect outliers using IQR method; if there are a large number of outliers, then the data is more skewed and may require transformation or use of non-linear models; if there are a small number of outliers, then the data is less skewed and more normally distributed which can be beneficial for linear regression models
    ########## if there are high number of outliers, do not transform them just yet to demo results with multiple models ###########
    q1 = frame[numericCols].quantile(0.25)
    q3 = frame[numericCols].quantile(0.75)
    iqr = q3 - q1
    lf = q1 - 1.5*iqr
    uf = q3 + 1.5*iqr

    # # frame having columns totalOutliers, outliersPercent, lowerOutliers, lowerOutliersPercent, upperOutliers, upperOutliersPercent for each numeric column
    outlierSummary = pd.DataFrame(index=numericCols, columns=['LowerOutliersCount', 'LowerOutliersPercent', 'UpperOutliersCount', 'UpperOutliersPercent', 'TotalOutliersCount', 'TotalOutliersPercent', 'Q1', 'Q3', 'IQR', 'LowerFence', 'UpperFence'])
    outlierSummary['Q1'] = q1
    outlierSummary['Q3'] = q3
    outlierSummary['IQR'] = iqr
    outlierSummary['LowerFence'] = lf
    outlierSummary['UpperFence'] = uf

    lowerOutliersMask = frame[numericCols] < lf
    lowerOutliersCount = (lowerOutliersMask).sum()
    outlierSummary['LowerOutliersCount'] = lowerOutliersCount

    lowerOutliersPercent = round(lowerOutliersCount * 100 / len(frame), 2)
    outlierSummary['LowerOutliersPercent'] = lowerOutliersPercent

    upperOutliersMask = frame[numericCols] > uf
    upperOutliersCount = (upperOutliersMask).sum()
    outlierSummary['UpperOutliersCount'] = upperOutliersCount
    upperOutliersPercent = round(upperOutliersCount * 100 / len(frame), 2)
    outlierSummary['UpperOutliersPercent'] = upperOutliersPercent

    totalOutliersCount = lowerOutliersCount + upperOutliersCount
    totalOutliersPercent = round(totalOutliersCount * 100 / len(frame), 2)
    outlierSummary['TotalOutliersCount'] = totalOutliersCount
    outlierSummary['TotalOutliersPercent'] = totalOutliersPercent

    # outliers = frame[(lowerOutliersMask | upperOutliersMask).any(axis=1)]

    # return {"OutlierSummary": outlierSummary, "Outliers": outliers}
    return outlierSummary


sec_time = record_time(sec_time, END, time.time())

### EDA

In [ ]:
# track the total time for section
sec_time = record_time(None, START, time.time())
timings[DATA_LOAD] = sec_time

# turn CoW on to avoid SettingWithCpyWarning and make it all more predictable and intuitive
pd.options.mode.copy_on_write = True

# read the data
df = pd.read_csv("creditcard.csv")
df.columns = df.columns.str.strip().str.replace(r"\s+", " ", regex=True).str.replace(".", "_").str.upper() # strip leading/trailing spaces, replace multiple spaces with single space and replace dots with underscores in column names for better handling in code

# preserve original frame just in case needed and work with the copy
originalDf = df.copy()

#print shape
print(f"Shape of the data: {df.shape}")

# print top sample
print("Head of frame")
prettyPrint(df.head())

#print last sample
print("Tail of frame")
prettyPrint(df.tail())

# objectsToBeDeployed = [] # track the objects to be deployed such as imputer, encoder, transformer, scaler, and model to prevent missing any of them
categoricalColumns = ['CLASS']
targetColumn = 'CLASS'
# idColumn = 'ID'

sec_time = record_time(sec_time, END, time.time())

In [ ]:
# track the total time for section
eda_sec_time = record_time(None, START, time.time())
timings[EDA] = eda_sec_time # UTIL_FUNCS, DATA_LOAD, EDA

#show basic info about the data
basicInfo = getBasicInfo(df)
printInfo(basicInfo)

columnInfo = basicInfo['columnInfo']
print("\nColumns having null values:\n")
prettyPrint(columnInfo[columnInfo['Null %'] > 0].sort_values('Null %', ascending=False))

print("\nColumns having blank values:\n")
prettyPrint(columnInfo[columnInfo['Blank %'] > 0].sort_values('Blank %', ascending=False))

<div style="font-size: 14px">
No data missing. Since there are no nulls, no imputation would be needed for missing data.
</div>

In [ ]:
desc = describeData(df, excludeCols=categoricalColumns)
prettyPrint(desc.T)

In [ ]:
df.nunique()

In [ ]:
print("\nDescriptive statistics for catagorical features:")
printCatagoricalStats(df, includeColumns=categoricalColumns, leftAlignTextColumns=False)

<div style="font-size: 14px">
<ol>
    <li>
        Above stats indicate there is class imbalance with non-fraud transactions totaling to 99.83% while fradulent transactions are at 0.17%. Thus class balancing will need to be done before any model training.
    </li>
</ol>
</div>

In [ ]:
# check class distribution

valueCounts = df[targetColumn].value_counts()
plt.figure(figsize=(4,3))
sns.barplot(x=valueCounts.index, y=valueCounts.values)
plt.title('Class Distribution')
plt.xticks(rotation=0)
plt.show()
plt.close()


In [ ]:
# remove duplicates if any
lenBefore = len(df)
df = deduplicate(df, cols=df.columns.tolist(), debug=True) # keeps first for any duplicates
lenAfter = len(df)

print(f"\nTotal rows before deduplication: {lenBefore}")
print(f"Total rows after deduplication: {lenAfter}")
print(f"Total rows removed: {lenBefore - lenAfter}")

In [ ]:
# missing value imputation: none of features have missing values so no imputation/cleanup is needed.

In [ ]:
# No categorical variables need encoding as they are already numeric

In [ ]:
print("\nData description after missing value imputation, data corrections, and deduplication:\n")
desc = describeData(df, excludeCols=categoricalColumns)
prettyPrint(desc.T)

In [ ]:
print("\nDescriptive statistics for catagorical features after removing dupes:")
printCatagoricalStats(df, includeColumns=categoricalColumns, leftAlignTextColumns=False)

In [ ]:
# check if mean and median are close to each other for numeric columns to get an idea about the skewness of the data; if mean and median are close, then the data is less skewed and more normally distributed which can be beneficial for linear regression models; if mean and median are far apart, then the data is more skewed and may require transformation or use of non-linear models
dfSkewness = pd.DataFrame(round(desc.loc["mean"] - desc.loc["50%"], 6), columns=['Mean-Median Difference'])
dfSkewness["Pearson's Skewness Indicator"] = 3*dfSkewness['Mean-Median Difference'] / desc.loc["std"]   #Pearson's skewness coefficient which is a commonly used measure of skewness; it is the mean-median difference normalized by the standard deviation to make it scale invariant and allow comparison across different columns; values close to 0 indicate low skewness, while values further from 0 indicate higher skewness; positive values indicate right skewness and negative values indicate left skewness
dfSkewness["Bowley's Skewness Indicator"] = (desc.loc["75%"] + desc.loc["25%"] - 2*desc.loc["50%"]) / (desc.loc["75%"] - desc.loc["25%"])   #Bowley's skewness coefficient which is a commonly used measure of skewness; it is the interquartile range normalized by the standard deviation to make it scale invariant and allow comparison across different columns; values close to 0 indicate low skewness, while values further from 0 indicate higher skewness; positive values indicate right skewness and negative values indicate left skewness
dfSkewness["Skew"] = df.select_dtypes(include=["number"]).skew()
prettyPrint(dfSkewness.sort_values(ascending=False, by="Skew"))

columnsToCheck = desc.columns
# find the percent of values according to empirical rule (68-95-99.7) to get an idea about the distribution of the data; 
# if a large percent of values are within 1 std of the mean, then the data is more normally distributed which can be beneficial for 
# linear regression models; if a large percent of values are outside 1 std of the mean, then the data is less normally distributed and 
# may require transformation or use of non-linear models
empiricalRule = pd.DataFrame({
    '% Within 1 Std': round(((df[columnsToCheck] - desc.loc["mean"]).abs() <= desc.loc["std"]).mean() * 100, 2), # count of trues/total gives percent of values within 1 std of the mean; round to 2 decimal places for better readability
    '% Within 2 Std': round(((df[columnsToCheck] - desc.loc["mean"]).abs() <= 2*desc.loc["std"]).mean() * 100, 2),
    '% Within 3 Std': round(((df[columnsToCheck] - desc.loc["mean"]).abs() <= 3*desc.loc["std"]).mean() * 100, 2)
})

prettyPrint(empiricalRule.sort_values(ascending=False, by='% Within 1 Std'))

In [ ]:
# plot histogram + KDE to visualize the distribution of the data; if the histogram is bell-shaped and the KDE curve is close to a normal 
# distribution curve, then the data is more normally distributed which can be beneficial for linear regression models; if the histogram 
# is skewed and the KDE curve is far from a normal distribution curve, then the data is less normally distributed and may require 
# transformation or use of non-linear models

# plot a grid of histograms + KDEs for all numeric columns for better readability
numericCols = df.select_dtypes(include=['number']).columns.difference(categoricalColumns, sort=False)
colsCount = len(numericCols)
grid_col_count = 3
grid_row_count = (colsCount + grid_col_count - 1) // grid_col_count # calculate grid size for 3 columns per row
plt.figure(figsize=(14, grid_row_count * 2.5)) # set figure size based on grid size
for i, col in enumerate(numericCols):
    plt.subplot(grid_row_count, grid_col_count, i + 1)
    sns.histplot(df[col], kde=True)
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")
plt.tight_layout()
plt.show()
plt.close()


In [ ]:
# plot outliers using boxplot to visualize the presence of outliers in the data; if there are many outliers, then the data is more skewed 
# and may require transformation or use of non-linear models; if there are few or no outliers, then the data is less skewed and may be 
# more suitable for linear regression models
numericCols = df.select_dtypes(include=['number']).columns.difference(categoricalColumns, sort=False)
grid_row_count = (len(numericCols) + 2) // 3 # calculate grid size for 3 columns per row
plt.figure(figsize=(14, grid_row_count * 2)) # set figure size based on grid size
for i, col in enumerate(numericCols):
    plt.subplot(grid_row_count, 3, i + 1)
    sns.boxplot(x=df[col])
    plt.title(f"Boxplot of {col}")
    plt.xlabel(col)
plt.tight_layout()
plt.show()
plt.close()

In [ ]:
# detect outliers uing IQR method
outlierStats = detectOutliersIQR(df, numericCols.difference(categoricalColumns))
prettyPrint(outlierStats.sort_values(by='TotalOutliersPercent', ascending=False))

# print(f"\nOutlier records: total {len(outlierStats["Outliers"])} out of {len(df)} records ({100*len(outlierStats["Outliers"])/len(df)}%)\n")
# print(outlierStats["Outliers"].head())
# # prettyPrint(outlierStats["Outliers"], maxRows=10)



In [ ]:
z_scores = np.abs(zscore(df[numericCols]))
z_df = pd.DataFrame(z_scores, columns=numericCols)

print("Outliers based on z-score method (count of values with z-score > 3); this is useful when data is near normal which is not the case here so IQR method is more reliable")
prettyPrint((z_df > 3).sum().sort_values(ascending=False))

<div style="font-size:14px">
<pre>Skew() (third-moment skewness) uses following rules of thumb:
    <b>Value	                Interpretation</b>
        ~= 0 (-0.1 to +0.1)     Symmetric
        +/-0.1 to +/-0.3        Slight skew
        +/-0.3 to +/-0.5        Moderate skew
        > +/-0.5                Strong skew
        > +/-0.7                Very strong / extreme skew
</pre>

Based on findings of skewness (skew() method), 3-std thresholds, and outlier detection (IQR method due to skewness), columns with absolute skewness over 0.5 - which is majority of columns here - may need to be transformed to address skewness and outliers and then all numeric columns need to be scaled to bring on the same scale.

<b>However, since the features here are components of PCA which are already transformed, no further transformation is really required.</b>

Comparing Bowley's skewness score (quartile based, robust to outliers) with skew function (third-moment skewness, sensitive to outliers), it shows bulk of data is only moderately skewed and centrally distributed while there are extreme outliers lengthening the tail making overall data extremely left/right skewed.
<ol>
    <li><b>Highly Right Skewed</b>
        Following features are extremely right skewed given high positive skewness scores and 1-STD covering over 75% data (vs empirical 68%) indicating high kurtosis i.e., majority of data is closer to mean but with strong outliers pulling mean higher. This means that most values are about average or smaller but there are some values that are much higher. A long tail indicates there are many extreme outliers which can also be seen in the outlier detection plots.
        Transformation would lead to reduction in number of outliers alongwith treating skewness.
        <ul>
            <li>V3</li>
            <li>V9</li>
            <li>AMOUNT</li>
            <li>V28</li>
            <li>V7</li>
            <li>V21</li>
            <li>V6</li>
            <li>V10</li>
            <li>V4</li>
            <li>V26</li>
        </ul>
    </li>
    <li><b>High Left Skewed</b>
        Following features are extremely left skewed given high negative skewness scores and 1-STD covering over 75% data (vs empirical 68%) indicating high kurtosis i.e., majority of data is closer to mean but with strong outliers pulling mean lower. This means that most values are about average or larger but there are some values that are much smaller. A long tail indicates there are many extreme outliers which can also be seen in the outlier detection plots. 
        Transformation would lead to reduction in number of outliers alongwith treating skewness.
        <ul>
        <li>V8</li>
        <li>V23</li>
        <li>V2</li>
        <li>V17</li>
        <li>V1</li>
        <li>V5</li>
        <li>V12</li>
        <li>V20</li>
        <li>V14</li>
        <li>V16</li>
        <li>V27</li>
        <li>V24</li>
        </ul>
    </li>
</ol>
</div>
<div style="font-size: 14px">
    Above stats indicate considerable (>5% each) outliers were identified on the either side of distribution in above columns. Dropping those rows is not ok as it would result in over 10% of data to be deleted. Because they may represent valid extreme transaction patterns and PCA-transformed feature values, they were retained.
</div>
<br/>
<div>
    Even though majority of columns have high skewness, they will NOT be transformed since they are components from PCA which are already transformed and centered.
</div>

In [ ]:
# check correlation
correlationMatrix = df.select_dtypes(include=['number']).corr()

prettyPrint(correlationMatrix)

# remove upper triangle which is redundant
mask = np.triu(np.ones_like(correlationMatrix, dtype=bool))
corr_lower = correlationMatrix.mask(mask)

plt.figure(figsize=(20, 10))
sns.heatmap(corr_lower, annot=True, fmt=".2f", cmap='coolwarm', cbar=True)
plt.title("Correlation Matrix before handling skewness, outliers, and scaling")
plt.show()
plt.close()

In [ ]:
# flatten corr matrix and sort by corr for easy inference
corr_pairs = (
    corr_lower.where(np.tril(np.ones(corr_lower.shape), k=-1).astype(bool))
        .stack()
        .reset_index()
)
corr_pairs.columns = ['Feature 1', 'Feature 2', 'Correlation']
corr_pairs = corr_pairs.sort_values(by='Correlation', key=abs, ascending=False)

print("\nCorr of features with target:\n")
# get correlation of features with target variable from corr_pairs
corr_with_target = (
    correlationMatrix[targetColumn]
    .drop(targetColumn)
    .sort_values(key=abs, ascending=False)
)

prettyPrint(corr_with_target)

corrMask80 = corr_pairs["Correlation"].abs() >= 0.8
highCorrFeatures = corr_pairs[corrMask80]
print(f"\nFeatures with atleast 0.8 correlation score sorted by corr value: total {len(highCorrFeatures)}\n")
prettyPrint(highCorrFeatures, maxRows=None)

feature_corr = corr_pairs[corr_pairs["Feature 2"] != targetColumn]
# print("\nTop 10 highly correlated features:\n")
# prettyPrint(feature_corr.head(10))

print("\nTop 10 least correlated features:\n")
prettyPrint(feature_corr.tail(10))

print("\nAll correlations sorted by corr value:\n")
prettyPrint(corr_pairs, maxRows=10)



In [ ]:
# # box plots for each feature to target to see feature distribution across classes and presence of outliers in each class; 
# # this can help in deciding the feature transformations and models to be used; if there are distinct differences in feature distribution 
# # across classes, then the data is more likely to be classified well by linear models; if there is a lot of overlap in feature distribution 
# # across classes, then the data may require transformation or use of non-linear models
# # import seaborn as sns
# # import matplotlib.pyplot as plt

grid_col_count = 3
cols = df.columns.difference(categoricalColumns)
grid_row_count = (len(cols) + 2) // grid_col_count # calculate grid size for N columns per row
plt.figure(figsize=(14, grid_row_count * 4)) # set figure size based on grid size
for i, col in enumerate(cols):
    plt.subplot(grid_row_count, grid_col_count, i + 1)
    sns.boxplot(x=targetColumn, y=col, data=df)
    plt.title(f"{col} vs {targetColumn}")
    # plt.xlabel(col)
plt.tight_layout()
plt.show()
plt.close()

In [ ]:
# # count plots for each categorical feature to target to see feature distribution across classes and presence of outliers in each class; 
# # this can help in deciding the feature transformations and models to be used; if there are distinct differences in feature distribution 
# # across classes, then the data is more likely to be classified well by linear models; if there is a lot of overlap in feature distribution 
# # across classes, then the data may require transformation or use of non-linear models
# # import seaborn as sns
# # import matplotlib.pyplot as plt

# THIS DATASET HAS NO CATEGORICAL COLUMNS other than the target itself so there is nothing to analyse here

grid_col_count = 3
cols = pd.Index(categoricalColumns).difference([targetColumn])
grid_row_count = (len(cols) + 2) // grid_col_count # calculate grid size for N columns per row
plt.figure(figsize=(14, grid_row_count * 4)) # set figure size based on grid size
for i, col in enumerate(cols):
    plt.subplot(grid_row_count, grid_col_count, i + 1)
    sns.countplot(x=col, hue=targetColumn, data=df)
    plt.title(f"{col} vs {targetColumn}")
    # plt.xlabel(col)
plt.tight_layout()
plt.show()
plt.close()

In [ ]:
# Analyse data to see class-wise feature statistics including outlier counts and percentages
# columns to analyze (exclude target)
features = [col for col in df.columns if col != "CLASS"]

metrics = []

for feature in features:
    for cls in sorted(df["CLASS"].unique()):
        data = df.loc[df["CLASS"] == cls, feature]

        q1 = data.quantile(0.25)
        q2 = data.quantile(0.50)
        q3 = data.quantile(0.75)

        iqr = q3 - q1

        lower_fence = q1 - 1.5 * iqr
        upper_fence = q3 + 1.5 * iqr

        lower_outliers = (data < lower_fence).sum()
        upper_outliers = (data > upper_fence).sum()
        total_outliers = lower_outliers + upper_outliers

        metrics.append({
            "Feature": feature,
            "Class": cls,
            "Count": len(data),

            "Min": data.min(),
            "Q1": q1,
            "Median": q2,
            "Q3": q3,
            "Max": data.max(),

            "IQR": iqr,

            "Lower Fence": lower_fence,
            "Upper Fence": upper_fence,

            "Lower Outliers": lower_outliers,
            "Upper Outliers": upper_outliers,
            "Total Outliers": total_outliers,

            "Outlier %": round(total_outliers / len(data) * 100, 2)
        })


class_feature_stats = pd.DataFrame(metrics)

print("Class-wise feature statistics including outlier counts and percentages:\n")
prettyPrint(class_feature_stats)

In [ ]:
# Check the median shift between classes
median_difference = (
    df.groupby("CLASS")[features]
      .median()
      .T
)

median_difference["Absolute Difference"] = (
    median_difference[1] - median_difference[0]
).abs()

median_difference.sort_values(
    "Absolute Difference",
    ascending=False
)

<div style="font-size:14px">
    Based on above histogram+KDE, boxplots, overall and class-wise quartile/outlier detection tables, corr matrix, and scatter plots, few observations can be as follows:
    <ol>
        <li>
            No single feature shows a very strong correlation with the target, some features exhibit some variation across classes, there is significant overlap, indicating that classification likely depends on a combination of multiple features rather than any single dominant predictor. In other words, no single feature can fully predict fraud. This suggests the need for models capable of capturing multivariate and potentially non-linear relationships.
        </li>
        <li>
            Scale difference exist primarily because features V1-V28 are PCA components. Thus scaling may be beneficial for models sesitive to magnitude like logistic regression, SVM, KNN and ANN.
        </li>
        <li>
            PCA components have very low correlation with each other indicating PCA reduced feature dependency. Highest correlation with target class is observed for V17, V14, V12, V10, and V16 so they might come up indicators of fraud patterns.
        </li>
        <li>
            Fraud and normal transactions overlap significantly for all features indicating there is no single predictor of frauds. SO models that learn complex patterns might be more useful.
        </li>
        <li>
            Most features have a lower median shift (difference) from normal to fraud transactions with following being significant: V14, V12, V17, V3, V10, V16, Time, Amount. These features also have high correlation with target class which together indicate that as values for these features reduce liklihood of fraud increases.</br>
            Following features have noticable high median shift: V2, V4, V11.</br>
            Patterns of combinations of these features might be good indicators of fraud.
        </li>
    </ol>
</div>

In [ ]:
# Create quantile bins for amount
df["AMOUNT_BIN"] = pd.qcut(
    df["AMOUNT"],
    q=10,
    duplicates="drop"
)

# Uncomment to see bins and counts
# print("Bins and transaction counts in each bin:\n", df["AMOUNT_BIN"].value_counts().sort_index())

In [ ]:
# Get bin analysis
amount_bins = (
    df.groupby("AMOUNT_BIN", observed=True)
      .agg(
          min_amount=("AMOUNT", "min"),
          max_amount=("AMOUNT", "max"),
          avg_amount=("AMOUNT", "mean"),
          median_amount=("AMOUNT", "median"),
          transactions=("AMOUNT", "count"),
          percentage_of_transactions=("AMOUNT", 
                                      lambda x: len(x)/len(df)*100)
      )
      .reset_index()
)

prettyPrint(amount_bins)

In [ ]:
# Show distribution of transactions by class and amount bins

#Below uses the quantile bins created above for better readability and to see the distribution of transactions across different amount ranges and classes
plt.figure(figsize=(10,5))

sns.countplot(
    data=df,
    x="AMOUNT_BIN",
    hue="CLASS"
)

plt.xticks(rotation=20)
plt.xlabel("Amount Range")
plt.ylabel("Transaction Count")
plt.title("Transaction Distribution by Amount Range and Class")
plt.show()
plt.close()

In [ ]:
#### Fraud rate analysis by amount bins
amount_analysis = (
    df.groupby("AMOUNT_BIN", observed=True)
      .agg(
          total_transactions=("CLASS","count"),
          fraud_transactions=("CLASS","sum"),
          fraud_rate=("CLASS",
                      lambda x: x.mean()*100)
      )
      .reset_index()
)

prettyPrint(amount_analysis)

In [ ]:
# plot fraud rate by amount bins
plt.figure(figsize=(6,4))

sns.barplot(
    data=amount_analysis,
    x="AMOUNT_BIN",
    y="fraud_rate"
)

plt.xticks(rotation=20)
plt.ylabel("Fraud Rate (%)")
plt.xlabel("Amount Range")
plt.title("Fraud Rate by Transaction Amount Range")
plt.show()
plt.close()

<div style="font-size:14px">
Based on above amount analysis, following can be noted:
<ol>
    <li>
        Transaction amounts are highly right-skewed, with ~90% of transactions below ₹203, while the highest amount bin contains the long tail of high-value transactions
    </li>
    <li>
        Fraud transactions are spread throughout amount bins and the rate varies with maximum frauds happening in lowest amount range followed by highest amount range. This tells amount has some predictive signal. But it is alone not sufficient to predict fraud as fraud is not linearly increasing or decreasing across amount ranges.
    </li>
</ol>

</div>

In [ ]:
#### Perform time analysis
# Create time bins of 2 hours each to see the distribution of transactions across different time ranges and classes

# 2 hour window in seconds
time_window = 2 * 60 * 60   # 7200 seconds

# create bins from min to max time
time_bins = range(
    int(df["TIME"].min()),
    int(df["TIME"].max()) + time_window,
    time_window
)

# create time bin column
df["TIME_BIN"] = pd.cut(
    df["TIME"],
    bins=time_bins,
    include_lowest=True
)

#Uncomment to see the time bins and counts
# print(time_bins)
# print(df["TIME_BIN"].value_counts().sort_index())

In [ ]:
total_transactions = len(df)
total_frauds = df["CLASS"].sum()

# Calculate time distribution statistics for each time bin
time_distribution = (
    df.groupby("TIME_BIN", observed=False)
      .agg(
        #   min_time=("TIME", "min"),
        #   max_time=("TIME", "max"),
        #   avg_time=("TIME", "mean"),
        #   median_time=("TIME", "median"), # these avgs don't make sense for time bins
          transactions=("TIME", "count"),
          fraud_transactions=("CLASS", "sum")
      )
)

time_distribution["transaction_density_%"] = (
    time_distribution["transactions"]
    / total_transactions
    * 100
)

time_distribution["fraud_density_%"] = (
    time_distribution["fraud_transactions"]
    / total_frauds
    * 100
)

time_distribution["fraud_rate"] = (
    time_distribution["fraud_transactions"]
    / time_distribution["transactions"]
    * 100
)

time_distribution = time_distribution.reset_index()

prettyPrint(time_distribution)

In [ ]:
# Plot transaction density by time bins
plt.figure(figsize=(14,5))

sns.barplot(
    data=time_distribution,
    x="TIME_BIN",
    y="transaction_density_%"
)

plt.xticks(rotation=45)

plt.title(
    "Transaction Density by 2 Hour Time Window"
)

plt.xlabel("Time Window (seconds)")
plt.ylabel("Transaction Density (%)")

plt.show()
plt.close()

In [ ]:
# Plot fraud transaction density by time bins
plt.figure(figsize=(14,5))

sns.barplot(
    data=time_distribution,
    x="TIME_BIN",
    # y="fraud_transactions"
    y="fraud_density_%"
)

plt.xticks(rotation=45)

plt.title(
    "Fraud Transaction Density by 2 Hour Time Window"
)

plt.xlabel("Time Window (seconds)")
plt.ylabel("Fraud Transaction Density (%)")

plt.show()
plt.close()

In [ ]:
# plot fraud rate by time bins
plt.figure(figsize=(14,5))

sns.barplot(
    # data=time_fraud_analysis,
    data=time_distribution,
    x="TIME_BIN",
    y="fraud_rate"
)

plt.xticks(rotation=45)

plt.title(
    "Fraud Rate by 2 Hour Time Window"
)

plt.xlabel("Time Window (seconds)")
plt.ylabel("Fraud Rate (%)")

plt.show()
plt.close()

In [ ]:
plot_df = time_distribution.copy()

# Convert interval bins to strings for plotting
plot_df["TIME_BIN_LABEL"] = plot_df["TIME_BIN"].astype(str)

# Create numeric positions for cleaner x-axis handling
plot_df["BIN_NO"] = range(len(plot_df))


fig, ax1 = plt.subplots(figsize=(14,6))

# Bar plot: transaction density
sns.barplot(
    data=plot_df,
    x="BIN_NO",
    y="transaction_density_%",
    ax=ax1,
    alpha=0.6,
    label="Transaction Density (%)"
)

ax1.set_ylabel("Transaction Density (%)")
ax1.set_xlabel("Time Window (2 hour bins)")


# Second axis
ax2 = ax1.twinx()

# Fraud density line
sns.lineplot(
    data=plot_df,
    x="BIN_NO",
    y="fraud_density_%",
    marker="o",
    ax=ax2,
    label="Fraud Density (%)"
)

# Fraud rate line
sns.lineplot(
    data=plot_df,
    x="BIN_NO",
    y="fraud_rate",
    marker="o",
    ax=ax2,
    label="Fraud Rate (%)"
)

ax2.set_ylabel("Fraud Density / Fraud Rate (%)")


# Replace x labels with actual intervals
ax1.set_xticks(plot_df["BIN_NO"])
ax1.set_xticklabels(plot_df["TIME_BIN_LABEL"], rotation=45, ha="right")


plt.title("Transaction Density, Fraud Density and Fraud Rate by Time Window")
plt.tight_layout()
plt.show()
plt.close()

# record eda end time
eda_sec_time = record_time(eda_sec_time, END, time.time())

<div style="font-size:14px">
Time-based analysis shows a repeated pattern across the two-day transaction period. Similar transaction density and fraud-rate behaviour is observed in corresponding time windows on both days, suggesting that the relationship between transaction timing and fraud occurrence is not purely random. Fraud risk is relatively higher in the early part of each day (particularly around the second 2-hour window), after which the fraud rate decreases and remains comparatively stable for the rest of the day. This indicates that TIME may capture useful temporal behaviour patterns for fraud prediction.
</div>

### Model Training

In [ ]:
# track the total time for section
exp_time_start = time.time()
experiments_sec_time = record_time(None, START, exp_time_start)
timings[EXPERIMENTS] = experiments_sec_time

# track the total time for section
sec_time = record_time(None, START, exp_time_start)
timings[SPLIT] = sec_time # UTIL_FUNCS, DATA_LOAD, EDA, EXPERIMENTS, SPLIT

# Do x-y and train-test split before transformation to avoid data leakage and to get a more realistic estimate of model performance on 
# unseen data; also, transformations should be fit only on the training data and then applied to the test data to prevent data leakage 
# and ensure that the model is not biased by information from the test set during training

# Remove temporary columns added for EDA
df.drop(columns=["AMOUNT_BIN", "TIME_BIN"], inplace=True, errors="ignore")

# Split original frame with outliers and nothing treated. Copies of these will be used to be able to compare results of these datasets 
# with and without transformation and scaling
X = df.drop(targetColumn, axis=1)
y = df[targetColumn]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y) # use stratify to maintain class distribution in train and test sets

print(f"X_train shape {X_train.shape} and data type {type(X_train)}")
print(f"X_test shape {X_test.shape} and data type {type(X_test)}")
print(f"y_train shape {y_train.shape} and data type {type(y_train)}")
print(f"y_test shape {y_test.shape} and data type {type(y_test)}")

print("\nTrain class distribution")
print(y_train.value_counts(normalize=True) * 100)

print("\nTest class distribution")
print(y_test.value_counts(normalize=True) * 100)

sec_time = record_time(sec_time, END, time.time())

In [ ]:
# track the total time for section
sec_time = record_time(None, START, time.time())
timings[SCALING] = sec_time

# scale training data

scaler = StandardScaler()

# doing this returns numpy ndarray and the original feature names are stripped. This causes the model to be trained on names like 
# feature01, feature02 etc. When shipped as a pipeline, pipeline.predict fails complaining about different column names when exact 
# feature names are given when creating the input dataframe. Either do this or train entire pipeline during experimentation and ship that 
# pipeline to avoid any errors at all. See partial transforms below and their datatypes in output.
# X_train_scaled = scaler.fit_transform(X_train)
# X_test_scaled = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.fit_transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

X_train_part_scaled = X_train.copy()
scaler_part = StandardScaler() # make sure to NOT use the original scalar else its columns and entire learning would change causing all the model trainings to be errorneous
X_train_part_scaled[["TIME", "AMOUNT"]] = scaler_part.fit_transform(X_train[["TIME", "AMOUNT"]])
X_train_part_scaled = X_train_part_scaled.to_numpy() # convert to ndarray for better performance with sklearn

X_test_part_scaled = X_test.copy()
X_test_part_scaled[["TIME", "AMOUNT"]] = scaler_part.transform(X_test[["TIME", "AMOUNT"]])
X_test_part_scaled = X_test_part_scaled.to_numpy() # convert to ndarray for better performance with sklearn

print(f"X_train_scaled shape {X_train_scaled.shape} and data type {type(X_train_scaled)}")
print(f"X_test_scaled shape {X_test_scaled.shape} and data type {type(X_test_scaled)}")
print(f"X_test_scaled flags: {X_train_scaled.flags}")
print(f"X_train_part_scaled shape {X_train_part_scaled.shape} and data type {type(X_train_part_scaled)}")
print(f"X_test_part_scaled shape {X_test_part_scaled.shape} and data type {type(X_test_part_scaled)}")
print(f"X_test_part_scaled flags: {X_train_part_scaled.flags}")

# track the total time for section
sec_time = record_time(sec_time, END, time.time())

In [ ]:
# Define model name constants for uniform usage
LOGISTIC_REGRESSION = "Logistic Regression"
SVM = "Support Vector Machine"
KNN = "K-Nearest Neighbors"
DECISION_TREE = "Decision Tree"
RANDOM_FOREST = "Random Forest"
ENSEMBLE = "Ensemble"
GRADIENT_BOOSTING = "Gradient Boosting"
XG_BOOST = "XGBoost"
LIGHT_GBM = "LightGBM"
ANN = "Artificial Neural Network(ANN)"
LOGISTIC_REGRESSION_SCALED = "Logistic Regression Scaled"
SVM_SCALED = "Support Vector Machine Scaled"
KNN_SCALED = "K-Nearest Neighbors Scaled"
DECISION_TREE_SCALED = "Decision Tree Scaled"
RANDOM_FOREST_SCALED = "Random Forest Scaled"
ENSEMBLE_SCALED = "Ensemble Scaled"
GRADIENT_BOOSTING_SCALED = "Gradient Boosting Scaled"
XG_BOOST_SCALED = "XGBoost Scaled"
LIGHT_GBM_SCALED = "LightGBM Scaled"
ANN_SCALED = "Artificial Neural Network(ANN) Scaled"

# Create model registry to track all models
model_registry = {} #init empty model registry and build as the models are trained

In [ ]:
# Model training and eval functions

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)    

def evaluate_model(model, X, y, model_name=None, classification_thresholds=[0.5]):

    print("Evaluating model", model_name)
    start = time.time()

    # dictionary to hold performance metrics
    metrics = {}
    
    if "ann" not in model_name.lower():
        y_pred = model.predict(X)

        # prettyPrint(y_pred.head())
        print(f"Type of y_pred: {type(y_pred)}")

        predict_time = time.time() - start
        print(f"Actual prediction time: {predict_time:.2f} sec")

        start = time.time()

        # probability scores needed; ANN already returns probabilities
        # Important: If threshold tuning is not done, .predict gives the final classificaition which can be used for metrics and 
        # prediction from user input. If threshold tuning is however done, .predict should be avoided and probability function should be
        # used instead to get the probabilities. And then threshold comparision be done with probabilites to do classification giving 
        # y_pred. To avoid errors then create a custom predict funciton which simply finds probabilities using probability function and does
        # classification using threshold comparision. This function and the threshold should be deployed and used at the time of prediction
        # passing the threshold to predict. This eliminates errors and keeps results consistent during training and prediction!
        # predict(model, X, classification_threshold)
        # Above approach is same as what is done in ANN, see below.
        # Here threshold tuning is not being done but the probabilities are used only for AUC plotting.
        if hasattr(model, "predict_proba"):
            proba_func = "predict_proba"
            y_prob = model.predict_proba(X)[:,1]
        else:
            y_prob = model.decision_function(X)
            proba_func = "decision_function"

        proba_predict_time = time.time() - start
        print(f"Actual probability prediction time with {proba_func} probability function: {proba_predict_time:.2f} sec")

        metrics = {
            "confusion_matrix": confusion_matrix(y, y_pred),
            "accuracy": accuracy_score(y,y_pred),
            "precision": precision_score(y,y_pred,zero_division=0),
            "recall": recall_score(y,y_pred,zero_division=0),
            "f1": f1_score(y,y_pred,zero_division=0)
            }
    else:
        # ANN gives probabilities instead of direct classification. So threshold comparision is used for classification. That also allows
        # you to easily do threshold tuning. This should be formed into a separate method:
        # predict(model, X, classification_threshold)
        y_prob = model.predict(
            X,
            verbose=0
        ).flatten()
        
        predict_time = time.time() - start
        print(f"Actual prediction time: {predict_time:.2f} sec")

        # define the mtrics dictionary which would return the metrics for each threshold; sample below
        """{"0.5": {
            "confusion_matrix": confusion_matrix(y, y_pred),
            "accuracy": accuracy_score(y,y_pred),
            "precision": precision_score(y,y_pred,zero_division=0),
            "recall": recall_score(y,y_pred,zero_division=0),
            "f1": f1_score(y,y_pred,zero_division=0)
            }}"""

        # keep the default classification threshold same as the default for other models = 0.5. Then for best models try various 
        # thresholds to see if recall goes up and precision stays acceptable.
        for th in classification_thresholds:
            y_pred = (
                y_prob >= th
            ).astype(int)

            scores = {
            "confusion_matrix": confusion_matrix(y, y_pred),
            "accuracy": accuracy_score(y,y_pred),
            "precision": precision_score(y,y_pred,zero_division=0),
            "recall": recall_score(y,y_pred,zero_division=0),
            "f1": f1_score(y,y_pred,zero_division=0)
            }
            metrics[th] = scores

    return {
        "y_prob": y_prob, # return probabilities for plotting auc curves
        "roc_auc": roc_auc_score(y,y_prob),
        "pr_auc": average_precision_score(y,y_prob),
        "metrics": metrics
        }

def train_model(model_name, model_config, X_train, y_train, X_test, y_test, X_train_scaled, X_test_scaled, callbacks=None, validation_split=0.15):

    print(f"Training {model_name}")

    model = model_config["model"]

    if model_config["data"] == "scaled":
        Xtrain = X_train_scaled
        Xtest = X_test_scaled
    else:
        Xtrain = X_train
        Xtest = X_test

    imbalance = model_config["imbalance"]

    start = time.time()
    if "ann" in model_name.lower():
        if callbacks and len(callbacks) > 0:
            history = model.fit(
            Xtrain,
            y_train,
            validation_split=validation_split,
            epochs=100,
            batch_size=512,
            verbose=1,
            class_weight = imbalance[1] if imbalance else None,
            callbacks=callbacks,
            shuffle=True
        )
        else:
            history = model.fit(
            Xtrain,
            y_train,
            validation_split=validation_split,
            epochs=100,
            batch_size=512,
            verbose=1,
            class_weight = imbalance[1] if imbalance else None,
            shuffle=True
        )
    elif imbalance and imbalance[0] == "sample_weight":
        model.fit(
            Xtrain,
            y_train,
            sample_weight=imbalance[1]
        )
        history = None # history is available only for TensorFlow/Keras models and not scikit-learn models
    else:
        model.fit(
            Xtrain,
            y_train # class_weight not needed here since this is part of model definition itself
        )
        history = None # history is available only for TensorFlow/Keras models and not scikit-learn models

    actual_train_time = time.time() - start
    print(f"Completed training {model_name}; Actual training time: {actual_train_time:.2f} sec")

    return {
    "model": model,
    "history": history,
    "scaling": model_config["data"],
    "xtrain": Xtrain,
    "xtest": Xtest,
    "ytrain": y_train,
    "ytest": y_test
    }

def pivot_results(results: pd.DataFrame):

    # Pivot train/test metrics into separate columns
    metrics_table = (
        results
        .pivot(
            index=["model"],
            columns="eval_type",
            values=[
                "data",
                "accuracy",
                "precision",
                "recall",
                "f1",
                "roc_auc",
                "pr_auc"
            ]
        )
    )

    # reorder MultiIndex columns: Train before Test
    metrics_table = metrics_table.reindex(
    columns=pd.MultiIndex.from_product(
        [
            metrics_table.columns.levels[0],
            ["Train", "Test"]
        ]
    )
)

    return metrics_table

def create_model_config():

    return {
        "model": None, # actual trained model instance
        "steps": [], # add pipeline steps
        "imbalance": None, # this is a tuple like ("scale_pos_weight", actual_weight) or ("weights", actual_weight) or None
        "metrics": {"train": None, "test": None}, # these hold the train and test metrics: {'confusion_matrix': [<actual matrix here>]), 'accuracy': xx.xx, 'precision': xx.xx, 'recall': xx.xx, 'f1': xx.xx}
        "data": "raw", # raw or scaled
        "features": None # actual features used in training
    }

def train_and_eval_model(model_registry, model_registry_key, model_name, model, imbalance, X_train, y_train, X_test, y_test, X_train_scaled, X_test_scaled, callbacks=None, scaled_data=False, scaler=None, validation_split=0.15, classification_thresholds=[0.5]):

    model_registry[model_registry_key] = create_model_config()
    model_config = model_registry[model_registry_key]

    model_config["model"] = model
    if(scaled_data):
        model_config["data"] = "scaled"
        model_config["steps"].append(scaler)
    model_config["steps"].append(model)
    model_config["imbalance"] = imbalance
    model_config["features"] = X_train.columns

    start = time.time()

    # train model
    model_training_info = train_model(model_registry_key, model_config, X_train, y_train, X_test, y_test, X_train_scaled, X_test_scaled, callbacks, validation_split)

    train_time = time.time() - start
    print(f"Training Time : {train_time:.2f} sec")

    if(model_registry_key == KNN_SCALED):
        print("Requested algorithm  :", model.algorithm)
        print("Actual fit method    :", model._fit_method)
        print("Effective metric     :", model.effective_metric_)
        print("Effective metric params:", model.effective_metric_params_)
        # print("Feature datatypes    :", X_train_scaled.dtypes)
        print("Training flags       :", X_train_scaled.flags)    
        print("Features seen by model:", model.n_features_in_)

    start = time.time()

    # eval model on training data
    eval_results_train = evaluate_model(model_training_info["model"], model_training_info["xtrain"], model_training_info["ytrain"], model_registry_key, classification_thresholds)

    predict_time_train = time.time() - start
    print(f"Training Prediction Time : {predict_time_train:.2f} sec")

    start = time.time()

    # eval model on test data
    eval_results_test = evaluate_model(model_training_info["model"], model_training_info["xtest"], model_training_info["ytest"], model_registry_key, classification_thresholds)
    
    predict_time_test = time.time() - start
    print(f"Testing Prediction Time : {predict_time_test:.2f} sec")

    if "ann" not in model_registry_key.lower():
        # get training metrics
        metrics = eval_results_train["metrics"]
        model_config["metrics"]["train"] = metrics

        # Convert the metrics dictionary to dataframe eliminating confusion_matrix
        mets = {k: v for k, v in metrics.items() if k != "confusion_matrix"}
        mets["roc_auc"] = eval_results_train["roc_auc"]
        mets["pr_auc"] = eval_results_train["pr_auc"]

        train_result = pd.DataFrame(mets, [0])
        train_result.insert(0, "model", model_name, allow_duplicates=True)
        train_result.insert(1, "data", model_config["data"], allow_duplicates=True)
        train_result.insert(2, "eval_type", "Train", allow_duplicates=True)
        train_result["num_of_epochs"] = len(model_training_info["history"].history["loss"]) if "ann" in model_registry_key.lower() else None

        # get testing metrics
        metrics = eval_results_test["metrics"]
        model_config["metrics"]["test"] = metrics

        # Convert the metrics dictionary to dataframe eliminating confusion_matrix
        mets = {k: v for k, v in metrics.items() if k != "confusion_matrix"}
        mets["roc_auc"] = eval_results_test["roc_auc"]
        mets["pr_auc"] = eval_results_test["pr_auc"]

        test_result = pd.DataFrame(mets, [0])
        test_result.insert(0, "model", model_name, allow_duplicates=True)
        test_result.insert(1, "data", model_config["data"], allow_duplicates=True)
        test_result.insert(2, "eval_type", "Test", allow_duplicates=True)
        test_result["num_of_epochs"] = None

        consolidatedDf = pd.concat([train_result, test_result], ignore_index=True)
    else:
        eval_metrics_train = eval_results_train["metrics"]
        model_config["metrics"]["train"] = eval_metrics_train

        eval_metrics_test = eval_results_test["metrics"]
        model_config["metrics"]["test"] = eval_metrics_test

        consolidatedDf = pd.DataFrame()

        for th in classification_thresholds:
            
            # get training metrics
            metrics = eval_metrics_train[th]

            # Convert the metrics dictionary to dataframe eliminating confusion_matrix
            mets = {k: v for k, v in metrics.items() if k != "confusion_matrix"}
            mets["roc_auc"] = eval_results_train["roc_auc"]
            mets["pr_auc"] = eval_results_train["pr_auc"]

            train_result = pd.DataFrame(mets, [0])
            train_result.insert(0, "model", f"{model_name}-threshold-{th}", allow_duplicates=True)
            train_result.insert(1, "data", model_config["data"], allow_duplicates=True)
            train_result.insert(2, "eval_type", "Train", allow_duplicates=True)
            train_result["num_of_epochs"] = len(model_training_info["history"].history["loss"]) if "ann" in model_registry_key.lower() else None

            # get testing metrics
            metrics = eval_metrics_test[th]

            # Convert the metrics dictionary to dataframe eliminating confusion_matrix
            mets = {k: v for k, v in metrics.items() if k != "confusion_matrix"}
            mets["roc_auc"] = eval_results_test["roc_auc"]
            mets["pr_auc"] = eval_results_test["pr_auc"]

            test_result = pd.DataFrame(mets, [0])
            test_result.insert(0, "model", f"{model_name}-threshold-{th}", allow_duplicates=True)
            test_result.insert(1, "data", model_config["data"], allow_duplicates=True)
            test_result.insert(2, "eval_type", "Test", allow_duplicates=True)
            test_result["num_of_epochs"] = None

            consolidatedDf = pd.concat([consolidatedDf, train_result, test_result], ignore_index=True)
    
    return {
        "results": consolidatedDf, 
        "model_training_info": model_training_info,
        "y_prob": {"train": eval_results_train["y_prob"], "test": eval_results_test["y_prob"]}
        }


# Create the baseline ANN
def build_baseline_ann(dropout=False):

    if dropout:
        model = Sequential([
            Dense(
                32,
                activation="relu",
                input_shape=(30,)
            ),
            Dropout(0.3),

            Dense(
                16,
                activation="relu"
            ),
            Dropout(0.2),

            Dense(
                1,
                activation="sigmoid"
            )
        ])
    else:
        model = Sequential([
            Dense(
                32,
                activation="relu",
                input_shape=(30,)
            ),

            Dense(
                16,
                activation="relu"
            ),

            Dense(
                1,
                activation="sigmoid"
            )
        ])

    model.compile(

        optimizer=Adam(
            learning_rate=0.001
        ),

        loss="binary_crossentropy",

        metrics=[
            "accuracy"
        ]
    )

    return model

def build_tuned_ann_model(hp):
    """Create the ANN architecture for hyperparameter tuning"""

    model = Sequential()

    # Select number of layers to have (max 3)
    num_layers = hp.Int(
    "num_layers",
    1,
    3
)

    # add the selected number of hidden layers
    for i in range(1, num_layers+1):

        units = hp.Choice(
            f"units_{i}",
            [16, 32, 64]
        )

        activation = hp.Choice(
            f"activation_{i}",
            ["relu", "leakyrelu"]
        )

        if(i == 1):
            model.add(
                    Dense(
                        units,
                        input_shape=(30,)
                    )
                )
        else:
            model.add(
                Dense(units)
            )

        if activation == "relu":
            model.add(ReLU())
        else:
            model.add(LeakyReLU())

        #### Not using dropout as early broad experiments already showed ann performing better without dropouts
        # # ---------- Dropout ----------
        # dropout_rate = hp.Choice(
        #     f"dropout_{i}",
        #     [0.0, 0.1, 0.2, 0.3]
        # )
        # if dropout_rate > 0:
        #     model.add(Dropout(dropout_rate))

    # ---------- Output ----------
    model.add(
        Dense(
            1,
            activation="sigmoid"
        )
    )

    learning_rate = hp.Choice(
        "learning_rate",
        [0.001, 0.0005, 0.0001]
    )

    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            AUC(name="roc_auc"),
            AUC(curve="PR", name="pr_auc")
        ]
    )

    return model

def build_model_tuner(model, hyperparams, nIterations, nJobs=-1, randSatate=42, cv=5):
    tuner = RandomizedSearchCV(
        estimator=model,
        param_distributions=hyperparams,
        n_iter=nIterations,
        scoring="average_precision", # scikit learn scoring metric for PR-AUC
        cv=cv,
        verbose=3,
        random_state=randSatate,
        n_jobs=nJobs, 
        refit=False # do not internally retrain the best estimator but do it explicitly with best params for consistency, logging and 
                    # building comparision table using the standard run_experiment function written in this project
    )

    return tuner

def plot_loss(history, title):

    plt.figure(figsize=(5,3))

    plt.plot(
        history.history["loss"],
        label="Training Loss"
    )

    if "val_loss" in history.history:
        plt.plot(history.history["val_loss"], label="Validation Loss")

    plt.title(title)
    plt.xlabel("Epoch")
    plt.ylabel("Binary Crossentropy")
    plt.legend()
    plt.grid(True)
    plt.show()

def plot_accuracy(history, title):

    plt.figure(figsize=(5,3))

    plt.plot(
        history.history["accuracy"],
        label="Training Accuracy"
    )

    if "val_accuracy" in history.history:
        plt.plot(
            history.history["val_accuracy"],
            label="Validation Accuracy"
        )

    plt.title(title)
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.grid(True)
    plt.show()

# y_true are the actual labels from train/test set whichever is being plotted
# y_prob is probability output of 1. model.predict for ANN 2. predict_proba or decision_function for other models
def plot_roc_curve(y_train, y_test, y_prob_train, y_prob_test, roc_auc_train, roc_auc_test, title="ROC Curve"):

    plt.figure(figsize=(5, 4))

    #Plot training AUC
    fpr, tpr, _ = roc_curve(y_train, y_prob_train)
    
    # plt.subplot(1, 2, 1)
    plt.plot(
        fpr,
        tpr,
        linewidth=2,
        label=f"ROC-AUC = {roc_auc_train:.4f}"
    )

    #Plot test AUC
    fpr, tpr, _ = roc_curve(y_test, y_prob_test)
    
    # plt.subplot(1, 2, 2)
    plt.plot(
        fpr,
        tpr,
        linewidth=2,
        label=f"ROC-AUC = {roc_auc_test:.4f}"
    )

    plt.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        linewidth=1
    )

    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    # plt.title(f"{title} (Test)")
    plt.title(f"{title} (Train vs Test)")
    plt.legend(loc="lower right")
    plt.grid(True)

    plt.tight_layout()
    plt.show()

def plot_pr_curve(y_train, y_test, y_prob_train, y_prob_test, pr_auc_train, pr_auc_test, title="Precision-Recall Curve"):

    plt.figure(figsize=(5, 4))

    # Plot training AUC
    precision_train, recall_train, _ = precision_recall_curve(
        y_train,
        y_prob_train
    )

    # plt.subplot(1, 2, 1)
    plt.plot(
        recall_train,
        precision_train,
        linewidth=2,
        label=f"PR-AUC = {pr_auc_train:.4f}"
    )

    # Plot testing AUC
    precision_test, recall_test, _ = precision_recall_curve(
        y_test,
        y_prob_test
    )
    # plt.subplot(1, 2, 2)
    plt.plot(
        recall_test,
        precision_test,
        linewidth=2,
        label=f"PR-AUC = {pr_auc_test:.4f}"
    )

    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"{title} (Train vs Test)")
    plt.legend(loc="lower left")
    plt.grid(True)

    plt.tight_layout()
    plt.show()

def run_experiment(model_registry, model_registry_key, model_name, model, imbalance, X_train, y_train, X_test, y_test, X_train_scaled, X_test_scaled, unified_metrics = None, add_to_unified_metrics=True, plot_auc=True, callbacks=None, scaled_data=False, scaler=None, validation_split=0.15, classification_thresholds=[0.5], print_unified_metrics = True):
    
    results = train_and_eval_model(model_registry, model_registry_key, model_name, model, imbalance, X_train, y_train, X_test, y_test, X_train_scaled, X_test_scaled, callbacks, scaled_data, scaler, validation_split, classification_thresholds)

    model_results_df = results["results"]
    print(f"Evaluation metrics for {model_name} model:")
    prettyPrint(model_results_df)
    
    model_config = model_registry[model_name]
    metrics = model_config["metrics"]

    if "ann" not in model_name.lower():
        conf_matrix = metrics["train"]["confusion_matrix"]
        print("Training confusion matrix")
        prettyPrint(pd.DataFrame(conf_matrix[::-1,::-1], index=["Actual Positive(Fraud)","Actual Negative"], columns=["Predicted Positive(Fraud)","Predicted Negative"]))
        conf_matrix = metrics["test"]["confusion_matrix"]
        print("Testing confusion matrix")
        prettyPrint(pd.DataFrame(conf_matrix[::-1,::-1], index=["Actual Positive(Fraud)","Actual Negative"], columns=["Predicted Positive(Fraud)","Predicted Negative"]))
    else:
        for th in classification_thresholds:
            conf_matrix = metrics["train"][th]["confusion_matrix"]
            print(f"Training confusion matrix for {model_name}-threshold-{th}")
            prettyPrint(pd.DataFrame(conf_matrix[::-1,::-1], index=["Actual Positive(Fraud)","Actual Negative"], columns=["Predicted Positive(Fraud)","Predicted Negative"]))
            conf_matrix = metrics["test"][th]["confusion_matrix"]
            print(f"Testing confusion matrix for {model_name}-threshold-{th}")
            prettyPrint(pd.DataFrame(conf_matrix[::-1,::-1], index=["Actual Positive(Fraud)","Actual Negative"], columns=["Predicted Positive(Fraud)","Predicted Negative"]))

    if(add_to_unified_metrics):
        unified_metrics = pd.concat([(pd.DataFrame() if unified_metrics is None else unified_metrics), model_results_df], ignore_index=True)
        
        if(print_unified_metrics):
            print("Unified metrics: ")
            prettyPrint(unified_metrics)

    if(plot_auc):
        train_result = model_results_df[model_results_df["eval_type"] == "Train"]
        test_result = model_results_df[model_results_df["eval_type"] == "Test"]
        y_prob = results["y_prob"]
        
        plot_roc_curve(y_train, y_test, y_prob["train"], y_prob["test"], train_result["roc_auc"].iloc[0], test_result["roc_auc"].iloc[0])
        plot_pr_curve(y_train, y_test, y_prob["train"], y_prob["test"], train_result["pr_auc"].iloc[0], test_result["pr_auc"].iloc[0])

    return results, unified_metrics


In [ ]:
# Define class balancing strategy: Since the dataset is highly imbalanced (0.17% minority), we will use class weights to penalize the 
# model for misclassifying the minority class (penalize false negatives to catch more frauds/imrpvoe fraud recall). 
# This will help the model to focus more on correctly classifying the minority class.
# SMOTE is not being used since we do not want a large number of synthetic fraud samples which can lead to overfitting and may 
# misrepresent the actual fraud distribution.
# Also weighted learning is supported by tree ensembles and ANN

from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights = dict(
    zip(classes, weights)
)

print("\nClass weights\n", class_weights)

In [ ]:

# define a dataframe that has metrics of all the models
unified_metrics = pd.DataFrame()


In [ ]:
# track the total time for section
sec_time = record_time(None, START, time.time())
timings[LOGISTIC_REGRESSION] = sec_time

# Create LogReg model and apply weights
model = lm.LogisticRegression(
    class_weight=class_weights,
    random_state=42
)

metrics, unified_metrics = run_experiment(model_registry, LOGISTIC_REGRESSION, LOGISTIC_REGRESSION, model, ("class_weights", class_weights), X_train, y_train, X_test, y_test, X_train_scaled, X_test_scaled, unified_metrics = unified_metrics, add_to_unified_metrics=True, plot_auc=True, scaler=scaler)

# track the total time for section
sec_time = record_time(sec_time, END, time.time())

In [ ]:
# track the total time for section
sec_time = record_time(None, START, time.time())
timings[LOGISTIC_REGRESSION_SCALED] = sec_time

# Create LogReg model and apply weights
model = lm.LogisticRegression(
    class_weight=class_weights,
    random_state=42
)

metrics, unified_metrics = run_experiment(model_registry, LOGISTIC_REGRESSION_SCALED, LOGISTIC_REGRESSION_SCALED, model, ("class_weights", class_weights), X_train, y_train, X_test, y_test, X_train_scaled, X_test_scaled, unified_metrics = unified_metrics, add_to_unified_metrics=True, plot_auc=True, scaled_data=True, scaler=scaler)

# track the total time for section
sec_time = record_time(sec_time, END, time.time())

In [ ]:
#### SVM is taking too long to train it never returns so it will not be considered for further experimentation
# # Create SVM model and apply weights
# model = svm.SVC(
#     class_weight=class_weights,
#     random_state=42,
#     probability=True
# )

# # model_results = train_and_eval_model(model_registry, SVM, model, ("class_weights", class_weights), X_train, y_train, X_test, y_test, X_train_scaled, X_test_scaled)
# # print(f"Evaluation metrics for {SVM} model:")
# # prettyPrint(model_results)

# # metricsdf = pd.concat([metricsdf, model_results], ignore_index=True)
# # print("Unified metrics: ")
# # prettyPrint(metricsdf)

# # prettyPrint(metricsdf.pivot(
# #             index=["model"],
# #             columns="eval_type",
# #             values=[
# #                 # "data",
# #                 "accuracy",
# #                 "precision",
# #                 "recall",
# #                 "f1",
# #                 "roc_auc",
# #                 "pr_auc"
# #             ]
# #         ))

# model = svm.SVC(
#     class_weight=class_weights,
#     random_state=42,
#     probability=True
# )

# results, unified_metrics = run_experiment(model_registry, SVM, SVM, model, ("class_weights", class_weights), X_train, y_train, X_test, y_test, X_train_scaled, X_test_scaled, unified_metrics = unified_metrics, add_to_unified_metrics=True, plot_auc=True)

In [ ]:
# # #### Don't use the defaults here as that takes a lot of time. Use the optimized one which gives exactly the same result but runs much
# # # faster.
# # # Create KNN model: try with default params to see how long does it take; train would be minimal as it is a lazy learner and simply copies 
# # # the data; thus prediction time would be larger as all the computation is moved to prediction time.
# # """Default params:
# # KNeighborsClassifier(
# #     n_neighbors=5,
# #     weights='uniform',
# #     algorithm='auto',
# #     leaf_size=30,
# #     p=2,
# #     metric='minkowski'
# # )"""

# # """A note on weights:
# # For class imbalanced classification, the models must be able to give different weights to different classes with higher weight to minority
# # class to penalize misclassification of minority more than majority i.e., not being able to identify the minority class (fraud). This beats 
# # the purpose of classifier. To handle this problem, balancing needs to be done either by telling the model to give higher weight to minority
# # or by oversampling the minority to generate syntheticate samples (SMOTE) or by undersampling majority. Undersampling may remove majority
# # of data leading to loss of info and poor training/prediction. Oversampling may lead to overfitting and may not represent the actual
# # distribution of fraud. Thus class weights is best approach.

# # However, KNN does not support class weights as there is no learning really happening; it simply copies the data during learning. And then
# # at prediction time, it finds the nearest neighbors and odes a majority voting. The weight parameter here for KNN is the weight of this 
# # voting and not class weights. Here weight could be uniform or distance based. Uniform means all neighbors have equal weight in voting
# # irrespective of their distance from the test point. Distance weighting means closer neighbors have higher weight in voting than farther
# # neighbors. Both can still misclassify minority as it is stil using majority voting.

# # The weights in other models are class weights meaning minority class has higher weight than majority class to prenalize misclassification 
# # of minority. Most models support weights, gradient boosting support sample weights instead of class weights, and XGB and LGBM support 
# # scale_pos_weight i.e., scale positive weight which means the weight of positive class (minority-fraud) is scaled up to penalize 
# # misclassification of minority more than majority.
# # """

# # Create LogReg model and apply weights; Don't run with defaults for its to slow
# model = ne.KNeighborsClassifier()
# model_results, unified_metrics = run_experiment(model_registry, KNN, KNN, model, None, X_train, y_train, X_test, y_test, X_train_scaled, X_test_scaled, unified_metrics = unified_metrics, add_to_unified_metrics=True, plot_auc=True)

# # # model_results = train_and_eval_model(model_registry, KNN, model, None, X_train, y_train, X_test, y_test, X_train_scaled, X_test_scaled)
# # # print(f"Evaluation metrics for {KNN} model:")
# # # prettyPrint(model_results)

# # # # metricsdf = pd.concat([metricsdf, model_results], ignore_index=True)
# # # print("Unified metrics: ")
# # # prettyPrint(metricsdf)

# # # del model, model_results

In [ ]:
# track the total time for section
sec_time = record_time(None, START, time.time())
timings[KNN] = sec_time

# Create KNN model: try with optimized params
"""
    n_neighbors=5,
    algorithm="ball_tree",  # Faster search than brute force for medium-dimensional data like 30 features.
    leaf_size=50,           # Slightly larger leaves reduce tree traversal overhead and often improve performance on datasets of this size.
    metric="euclidean",     # Equivalent to the default Minkowski distance with p=2, but avoids a small amount of parameter handling overhead.
    weights="uniform",      # Default behavior and slightly faster than distance weighting.
    n_jobs=-1               # Uses all CPU cores which substantially speeds up prediction.
"""

model = ne.KNeighborsClassifier(n_neighbors=5,
    algorithm="ball_tree",
    leaf_size=50,
    metric="euclidean",
    weights="uniform",
    n_jobs=-1)
metrics, unified_metrics = run_experiment(model_registry, KNN, KNN, model, None, X_train, y_train, X_test, y_test, X_train_scaled, X_test_scaled, unified_metrics = unified_metrics, add_to_unified_metrics=True, plot_auc=True)

# track the total time for section
sec_time = record_time(sec_time, END, time.time())

In [ ]:
# track the total time for section
sec_time = record_time(None, START, time.time())
timings[KNN_SCALED] = sec_time # UTIL_FUNCS, DATA_LOAD, EDA, EXPERIMENTS, SPLIT

model = ne.KNeighborsClassifier(n_neighbors=5,
    algorithm="brute",
    leaf_size=50,
    metric="euclidean",
    weights="uniform",
    n_jobs=-1)
metrics, unified_metrics = run_experiment(model_registry, KNN_SCALED, KNN_SCALED, model, None, X_train, y_train, X_test, y_test, X_train_scaled, X_test_scaled, unified_metrics = unified_metrics, add_to_unified_metrics=True, plot_auc=True, scaled_data=True, scaler=scaler)

# track the total time for section
sec_time = record_time(sec_time, END, time.time())

Above KNN experiments show that for this dataset, KNN performed substantially faster on the raw features with ball_tree algo than on either fully or partially scaled features (only time and amount scaled). Changing the nearest-neighbor search algorithm ('ball_tree', 'kd_tree', 'brute') did not eliminate the slowdown, indicating that the scaled feature representation itself adversely affected KNN prediction performance. <br/><br/>
Default config for knn with raw data took about 10 mins while knn with scaled data (fully or partially) did not even return the predict call even after 10 mins with kd_tree and ball_tree algos. brute with fully scaled data took about 9.5 mins and with part-scaled data took over 9 mins.<br/><br/>
Recall is zero with raw data.

In [ ]:
# track the total time for section
sec_time = record_time(None, START, time.time())
timings[DECISION_TREE] = sec_time

# Create Decision Tree model and apply weights
model = tr.DecisionTreeClassifier(
    class_weight=class_weights,
    random_state=42
)
metrics, unified_metrics = run_experiment(model_registry, DECISION_TREE, DECISION_TREE, model, ("class_weights", class_weights), X_train, y_train, X_test, y_test, X_train_scaled, X_test_scaled, unified_metrics = unified_metrics, add_to_unified_metrics=True, plot_auc=True)

# track the total time for section
sec_time = record_time(sec_time, END, time.time())

In [ ]:
# track the total time for section
sec_time = record_time(None, START, time.time())
timings[DECISION_TREE_SCALED] = sec_time

# Create Decision Tree model and apply weights
model = tr.DecisionTreeClassifier(
    class_weight=class_weights,
    random_state=42
)
metrics, unified_metrics = run_experiment(model_registry, DECISION_TREE_SCALED, DECISION_TREE_SCALED, model, ("class_weights", class_weights), X_train, y_train, X_test, y_test, X_train_scaled, X_test_scaled, unified_metrics = unified_metrics, add_to_unified_metrics=True, plot_auc=True, scaled_data=True, scaler=scaler)

# track the total time for section
sec_time = record_time(sec_time, END, time.time())

In [ ]:
# track the total time for section
sec_time = record_time(None, START, time.time())
timings[RANDOM_FOREST] = sec_time

# Create Decision Tree model and apply weights
model = en.RandomForestClassifier(
    n_estimators=100,
    n_jobs=-1,
    random_state=42,
    class_weight="balanced"
)
metrics, unified_metrics = run_experiment(model_registry, RANDOM_FOREST, RANDOM_FOREST, model, None, X_train, y_train, X_test, y_test, X_train_scaled, X_test_scaled, unified_metrics = unified_metrics, add_to_unified_metrics=True, plot_auc=True)

# track the total time for section
sec_time = record_time(sec_time, END, time.time())

In [ ]:
# track the total time for section
sec_time = record_time(None, START, time.time())
timings[RANDOM_FOREST_SCALED] = sec_time

model = en.RandomForestClassifier(
    n_estimators=100,
    n_jobs=-1,
    random_state=42,
    class_weight="balanced"
)
metrics, unified_metrics = run_experiment(model_registry, RANDOM_FOREST_SCALED, RANDOM_FOREST_SCALED, model, None, X_train, y_train, X_test, y_test, X_train_scaled, X_test_scaled, unified_metrics = unified_metrics, add_to_unified_metrics=True, plot_auc=True, scaled_data=True, scaler=scaler)

# track the total time for section
sec_time = record_time(sec_time, END, time.time())

In [ ]:
# track the total time for section
sec_time = record_time(None, START, time.time())
timings[GRADIENT_BOOSTING] = sec_time

# Create Gradient Boosting model and apply sample weights as it does not support class weights
sample_weight = y_train.map(
    class_weights
)

model = en.GradientBoostingClassifier(
    n_estimators=100,
    # n_jobs=-1,
    random_state=42
)
metrics, unified_metrics = run_experiment(model_registry, GRADIENT_BOOSTING, GRADIENT_BOOSTING, model, ("sample_weight", sample_weight), X_train, y_train, X_test, y_test, X_train_scaled, X_test_scaled, unified_metrics = unified_metrics, add_to_unified_metrics=True, plot_auc=True)

# track the total time for section
sec_time = record_time(sec_time, END, time.time())

In [ ]:
# track the total time for section
sec_time = record_time(None, START, time.time())
timings[GRADIENT_BOOSTING_SCALED] = sec_time

model = en.GradientBoostingClassifier(
    n_estimators=100,
    # n_jobs=-1,
    random_state=42
)
metrics, unified_metrics = run_experiment(model_registry, GRADIENT_BOOSTING_SCALED, GRADIENT_BOOSTING_SCALED, model, ("sample_weight", sample_weight), X_train, y_train, X_test, y_test, X_train_scaled, X_test_scaled, unified_metrics = unified_metrics, add_to_unified_metrics=True, plot_auc=True, scaled_data=True, scaler=scaler)

# track the total time for section
sec_time = record_time(sec_time, END, time.time())

In [ ]:
# Calculate scale_pos_weight (positive class weight) for XGBoost/LightGBM which is the ratio of negative class to positive class in the 
# training data; this will help the model to focus more on correctly classifying the minority class.
scale_pos_weight = (
    y_train.value_counts()[0] /
    y_train.value_counts()[1]
)

print(f"\nPOS weights for XGBoost/LightGBM\n{scale_pos_weight} (type = {type(scale_pos_weight)})")

In [ ]:
# track the total time for section
sec_time = record_time(None, START, time.time())
timings[XG_BOOST] = sec_time

# Create Random Forest model and apply weights
model = xgb.XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    n_jobs=-1,
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight
)
metrics, unified_metrics = run_experiment(model_registry, XG_BOOST, XG_BOOST, model, None, X_train, y_train, X_test, y_test, X_train_scaled, X_test_scaled, unified_metrics = unified_metrics, add_to_unified_metrics=True, plot_auc=True)

# track the total time for section
sec_time = record_time(sec_time, END, time.time())

In [ ]:
# track the total time for section
sec_time = record_time(None, START, time.time())
timings[XG_BOOST_SCALED] = sec_time

model = xgb.XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    n_jobs=-1,
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight
)
metrics, unified_metrics = run_experiment(model_registry, XG_BOOST_SCALED, XG_BOOST_SCALED, model, None, X_train, y_train, X_test, y_test, X_train_scaled, X_test_scaled, unified_metrics = unified_metrics, add_to_unified_metrics=True, plot_auc=True, scaled_data=True, scaler=scaler)

# track the total time for section
sec_time = record_time(sec_time, END, time.time())

In [ ]:
# track the total time for section
sec_time = record_time(None, START, time.time())
timings[LIGHT_GBM] = sec_time

# Create LightGBM model and apply pos weights
model = lgb.LGBMClassifier(
    n_estimators=100,
    learning_rate=0.1,
    # max_depth=6,
    random_state=42,
    n_jobs=-1,
    # eval_metric="logloss",
    scale_pos_weight=scale_pos_weight, # class_weight="balanced"
    verbosity=-1
)
metrics, unified_metrics = run_experiment(model_registry, LIGHT_GBM, LIGHT_GBM, model, None, X_train, y_train, X_test, y_test, X_train_scaled, X_test_scaled, unified_metrics = unified_metrics, add_to_unified_metrics=True, plot_auc=True)

# track the total time for section
sec_time = record_time(sec_time, END, time.time())

In [ ]:
# track the total time for section
sec_time = record_time(None, START, time.time())
timings[LIGHT_GBM_SCALED] = sec_time

model = lgb.LGBMClassifier(
    n_estimators=100,
    learning_rate=0.1,
    # max_depth=6,
    random_state=42,
    n_jobs=-1,
    # eval_metric="logloss",
    scale_pos_weight=scale_pos_weight, # class_weight="balanced"
    verbosity=-1
)
metrics, unified_metrics = run_experiment(model_registry, LIGHT_GBM_SCALED, LIGHT_GBM_SCALED, model, None, X_train, y_train, X_test, y_test, X_train_scaled, X_test_scaled, unified_metrics = unified_metrics, add_to_unified_metrics=True, plot_auc=True, scaled_data=True, scaler=scaler)

# track the total time for section
sec_time = record_time(sec_time, END, time.time())

In [ ]:
# track the total time for section
sec_time = record_time(None, START, time.time())
timings[f"{ANN}"] = sec_time

model_registry_key = ANN
model = build_baseline_ann()
metrics, unified_metrics = run_experiment(model_registry, model_registry_key, model_registry_key, model, ("class_weights", class_weights), X_train, y_train, X_test, y_test, X_train_scaled, X_test_scaled, unified_metrics = unified_metrics, add_to_unified_metrics=True, plot_auc=True)

# track the total time for section
sec_time = record_time(sec_time, END, time.time())

model_history = metrics["model_training_info"]["history"]

num_of_epochs = len(model_history.history["loss"])
print(f"Number of epochs run for {model_registry_key} model: {num_of_epochs}")

plot_loss(
    model_history,
    model_registry_key
)

plot_accuracy(
    model_history,
    model_registry_key
)


In [ ]:
# track the total time for section
sec_time = record_time(None, START, time.time())
timings[f"{ANN_SCALED}"] = sec_time

model_registry_key = ANN_SCALED
model = build_baseline_ann()
metrics, unified_metrics = run_experiment(model_registry, model_registry_key, model_registry_key, model, ("class_weights", class_weights), X_train, y_train, X_test, y_test, X_train_scaled, X_test_scaled, unified_metrics = unified_metrics, add_to_unified_metrics=True, plot_auc=True, scaled_data=True, scaler=scaler)

# track the total time for section
sec_time = record_time(sec_time, END, time.time())

model_history = metrics["model_training_info"]["history"]

num_of_epochs = len(model_history.history["loss"])
print(f"Number of epochs run for {model_registry_key} model: {num_of_epochs}")

plot_loss(
    model_history,
    model_registry_key
)

plot_accuracy(
    model_history,
    model_registry_key
)

In [ ]:
patience=25

# track the total time for section
sec_time = record_time(None, START, time.time())
timings[f"{ANN}-Early Stopping-{patience}"] = sec_time

model_registry_key = f"{ANN}-Early Stopping-{patience}"

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=patience,
    min_delta=0.001,
    restore_best_weights=True,
    verbose=1
)

model = build_baseline_ann()
metrics, unified_metrics = run_experiment(model_registry, model_registry_key, model_registry_key, model, ("class_weights", class_weights), X_train, y_train, X_test, y_test, X_train_scaled, X_test_scaled, unified_metrics = unified_metrics, add_to_unified_metrics=True, plot_auc=True, callbacks=[early_stopping])

# track the total time for section
sec_time = record_time(sec_time, END, time.time())

model_history = metrics["model_training_info"]["history"]

num_of_epochs = len(model_history.history["loss"])
print(f"Number of epochs run for {model_registry_key} model: {num_of_epochs}")

plot_loss(
    model_history,
    model_registry_key
)

plot_accuracy(
    model_history,
    model_registry_key
)

In [ ]:
# track the total time for section
sec_time = record_time(None, START, time.time())
timings[f"{ANN_SCALED}-Early Stopping-{patience}"] = sec_time

patience=25
model_registry_key = f"{ANN_SCALED}-Early Stopping-{patience}"

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=patience,
    min_delta=0.001,
    restore_best_weights=True,
    verbose=1
)

model = build_baseline_ann()
metrics, unified_metrics = run_experiment(model_registry, model_registry_key, model_registry_key, model, ("class_weights", class_weights), X_train, y_train, X_test, y_test, X_train_scaled, X_test_scaled, unified_metrics = unified_metrics, add_to_unified_metrics=True, plot_auc=True, callbacks=[early_stopping], scaled_data=True, scaler=scaler)

# track the total time for section
sec_time = record_time(sec_time, END, time.time())

model_history = metrics["model_training_info"]["history"]

num_of_epochs = len(model_history.history["loss"])
print(f"Number of epochs run for {model_registry_key} model: {num_of_epochs}")

plot_loss(
    model_history,
    model_registry_key
)

plot_accuracy(
    model_history,
    model_registry_key
)

In [ ]:
# track the total time for section
sec_time = record_time(None, START, time.time())
timings[f"{ANN}-Early Stopping+Dropout-{patience}"] = sec_time

patience=25
model_registry_key = f"{ANN}-Early Stopping+Dropout-{patience}"

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=patience,
    min_delta=0.001,
    restore_best_weights=True,
    verbose=1
)

model = build_baseline_ann(dropout=True)
metrics, unified_metrics = run_experiment(model_registry, model_registry_key, model_registry_key, model, ("class_weights", class_weights), X_train, y_train, X_test, y_test, X_train_scaled, X_test_scaled, unified_metrics = unified_metrics, add_to_unified_metrics=True, plot_auc=True, callbacks=[early_stopping])

# track the total time for section
sec_time = record_time(sec_time, END, time.time())

model_history = metrics["model_training_info"]["history"]

num_of_epochs = len(model_history.history["loss"])
print(f"Number of epochs run for {model_registry_key} model: {num_of_epochs}")

plot_loss(
    model_history,
    model_registry_key
)

plot_accuracy(
    model_history,
    model_registry_key
)

In [ ]:
# track the total time for section
sec_time = record_time(None, START, time.time())
timings[f"{ANN_SCALED}-Early Stopping+Dropout-{patience}"] = sec_time # UTIL_FUNCS, DATA_LOAD, EDA, EXPERIMENTS, SPLIT

patience=25
model_registry_key = f"{ANN_SCALED}-Early Stopping+Dropout-{patience}"

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=patience,
    min_delta=0.001,
    restore_best_weights=True,
    verbose=1
)

model = build_baseline_ann(dropout=True)
metrics, unified_metrics = run_experiment(model_registry, model_registry_key, model_registry_key, model, ("class_weights", class_weights), X_train, y_train, X_test, y_test, X_train_scaled, X_test_scaled, unified_metrics = unified_metrics, add_to_unified_metrics=True, plot_auc=True, callbacks=[early_stopping], scaled_data=True, scaler=scaler)

# track the total time for section
sec_time = record_time(sec_time, END, time.time())

model_history = metrics["model_training_info"]["history"]

num_of_epochs = len(model_history.history["loss"])
print(f"Number of epochs run for {model_registry_key} model: {num_of_epochs}")

plot_loss(
    model_history,
    model_registry_key
)

plot_accuracy(
    model_history,
    model_registry_key
)

<div style="font-size: 14px; overflow: auto">
Above results for various ANNs indicate the following:
<ol>
    <li>
        <b>Raw vs Scaled Data</b>
        <div>
            <pre>
    <b>Raw ANN</b>
    The raw ANN models are highly unstable across different runs. Depending on random initialization and training, some runs predict almost all transactions as genuine (recall ≈ 0), some predict almost all transactions as fraud (recall ≈ 1), while others learn reasonably well.
    Some raw ANN runs resulted in extremely low PR-AUC (around 0.006), with recall close to 1 and precision around 0.0016, indicating that the model classified almost every transaction as fraud.<br/>
    <b>Scaled ANN</b>
    The scaled models are:
    Stable across runs.
    Consistently achieve PR-AUC over 0.75 and recall over 0.80 indicating better fraud detection though precision falls a bit leading to slightly higher false positives.<br/>
    <b>Conclusion</b>: This indicates that gradient descent struggles to optimize on unscaled features. Scaling is essential and produces much better results for ANN on this dataset.
        </pre>
        </div>
    </li>
    <li>
    <b>Regularization</b>
    Both early stopping and dropout are not helping in improving the results beyond the baseline scaled version. Early stopping maintained high recall compared to scaled baseline but PR-AUC and precisions reduced while dropout worsened the results.
    <pre>
    <b>Raw + Early Stopping</b>
    This becomes better than raw baseline but is still unstable with ROC-AUC around 0.81–0.89 and PR-AUC in range of 0.24-0.42.
    <b>Scaled + Early Stopping</b>
    Performance similar to scaled baseline with dropped precision. Early stopping reduced epochs to average 65 (from max 100).
    This again proves to be very stable and consistently high performing with balanced precision/recall between (0.29-0.39/0.81-0.85), ROC-AUC between (0.94-0.96) and PR-AUC between (0.73-0.76)
    <b>Raw + Dropout</b>
    This again collapsed with similar extreme 0 or 1 results.
    <b>Scaled + Dropout</b>
    Unlike the raw dropout model, which collapsed into predicting nearly all transactions as fraud, the scaled dropout model remained stable. However, although recall stayed high (≈0.84), precision and F1-score remained substantially lower than the baseline, indicating excessive false positives. This suggests dropout caused underfitting rather than improving generalization.
    <b>Conclusion</b>: Based on these experiments, additional regularization through early stopping or dropout did not improve performance over the baseline scaled ANN. Therefore, subsequent optimization focused on hyperparameter tuning of the baseline scaled architecture.
    </pre>
    </li>
    <li>
        <b>Underfitting vs Overfitting (Generalization)</b>
        <div>
            <pre>
    <b>Raw baseline</b>
    Due to different scales, there is huge variation in various runs and between train and test metrics. The raw models frequently exhibited unstable behaviour consistent with optimization failure, leading to poor generalization and, in some runs, near-random predictions.
    <b>Scaled baseline</b>
    Train PR-AUC = 0.92, test PR-AUC = 0.75, a gap of ~0.17 means there is some overfiting. This was tried to be improved with tuning.
    Recall (Train: 0.9726, Test: 0.8309), precision (Train: 0.4733, Test: 0.4041), and F1 (Train: 0.6368, Test: 0.5437) between train and test remain reasonably close. This is acceptable.
    <b>Conclusion</b>: This shows scaling leads to good results, looks stable and consistent, no severe overfitting meaning the model can generalize well on unseen data.
    </pre>
    </div>
    </li>
</ol>


</div>

In [ ]:
pivoted_results = pivot_results(unified_metrics)
sorted_results = pivoted_results.sort_values(by=[("pr_auc", "Test"), ("recall", "Test"), ("data", "Test")], ascending=[False, False, True])
prettyPrint(sorted_results)

### Perform hyperparameter tuning for seleted models

<b>Model Selection for Fine-tuning:</b><br/>
<div>
Models were primarily compared using PR-AUC, as it is the most appropriate metric for highly imbalanced binary classification problems. Recall, precision, and F1-score were then examined to assess the trade-off between fraud detection capability and false alarms. ROC-AUC was used as a secondary measure of ranking performance, while accuracy was reported for completeness but not used for model selection.
</div><br/>

<div>
Based on the baseline results, XGBoost and Random Forest achieved the highest PR-AUC among the classical machine learning models and were therefore selected for hyperparameter tuning. Although the scaled ANN was not among the top three models by PR-AUC, it was also selected for tuning as the best-performing deep learning model to enable a fair comparison between optimized machine learning and neural network approaches.
</div>

In [ ]:
# track the total time for section
sec_time = record_time(None, START, time.time())
timings[ANN_TUNING] = sec_time

#### Tune ANN using RandomSearch keras tuner
# Create the keras RandomSearch tuner
rsTuner = kt.RandomSearch(

    hypermodel=build_tuned_ann_model,
    objective=kt.Objective(
        "val_pr_auc",
        direction="max"
    ),
    max_trials=15,
    overwrite=True,
    directory="keras_tuner",
    project_name="credit_card_fraud_ann"

)

#### Not using early stopping since earlier broad experiments showed it does not perform well
# early_stop = EarlyStopping(

#     monitor="val_loss",
#     patience=patience,  # Use same patience level as used for various ANNs before fine-tuning
#     restore_best_weights=True,
#     min_delta=0.001

# )

# Run the RandomSearch tuner
rsTuner.search(

    X_train_scaled,
    y_train,
    validation_split=0.15,
    epochs=100,
    batch_size=512,
    class_weight=class_weights,
    # callbacks=[early_stop],
    verbose=1,
    shuffle=True

)

# track the total time for section
sec_time = record_time(sec_time, END, time.time())

# report the best parameters used by tuner to generate the best model
best_hp = rsTuner.get_best_hyperparameters(1)[0]
print("Best hyperparameters for tuned ANN model:\n", best_hp.values, type(best_hp))

# This is the tuned model trained the same way as baseline but with different random hyperparams
best_model = rsTuner.get_best_models(1)[0]

In [ ]:
# track the total time for section
sec_time = record_time(None, START, time.time())
timings["Retrain Tuned ANN"] = sec_time

# Train the best fine-tuned ANN model on entire training dataset for consistent comparision with other models
# No early stopping callback or dropouts as they were already shown to be not performing better than scaled baseline
# Validation split set to zero to train on full training set for consistnet comparision with other models

# create a fresh ANN model using best hyperparameters so that weights are randomly initialized and there is no impact of fine-tuned weights
# Then train on full training set

model = build_tuned_ann_model(best_hp)

# Train the ANN with best hyperparams with different thresholds as default threshold of 0.5 has very very poor precision
# If any of these configs don't improve the precision that would mean the baseline scaled model works the best
thresholds = [0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8]
model_registry_key = f"{ANN_SCALED}-Fine-tuned-4"
metrics, unified_metrics = run_experiment(model_registry, model_registry_key, model_registry_key, model, ("class_weights", class_weights), X_train, y_train, X_test, y_test, X_train_scaled, X_test_scaled, unified_metrics = unified_metrics, add_to_unified_metrics=True, plot_auc=True, callbacks=None, scaled_data=True, scaler=scaler, validation_split=0, classification_thresholds=thresholds, print_unified_metrics=False)

model_history = metrics["model_training_info"]["history"]

num_of_epochs = len(model_history.history["loss"])
print(f"Number of epochs run for {model_registry_key} model: {num_of_epochs}")

# track the total time for section
sec_time = record_time(sec_time, END, time.time())

<div style="overflow: auto">
<b>Final ANN model selection</b>
Based on various experiments conducted on ANN architectures and hyperparameter tuning, the tuned scaled model seems to be performing the best. As of last test, following parameters were returned by tuner as the best:

{'num_layers': 1, 'units_1': 64, 'activation_1': 'relu', 'learning_rate': 0.001, 'units_2': 64, 'activation_2': 'leakyrelu'}
<!-- {'num_layers': 2, 'units_1': 32, 'activation_1': 'relu', 'learning_rate': 0.0001, 'units_2': 16, 'activation_2': 'relu'} -->
<!-- {'num_layers': 2, 'units_1': 64, 'activation_1': 'relu', 'learning_rate': 0.001, 'units_2': 32, 'activation_2': 'relu', 'units_3': 16, 'activation_3': 'relu'} -->


At a classification threshold of 0.5 (default), the tuned model performed better than the baseline scaled ANN architecture with two layers of 32 and 16 neurons respectively. This indicates that the model performance improved through hyperparameter tuning at default threshold. This tuned ANN model was finally selected for comparision with other scikit learn models to keep the comparision consisitent as other models did not recieve the threshold fine tuning.

After selecting the tuned ANN, an additional threshold sensitivity analysis was performed to study the precision-recall trade-off. Increasing the threshold from 0.5 to 0.8 improved precision from 0.4427 to 0.5858 and F1-score from 0.5742 to 0.6823 while maintaining same recall (0.817) on the test set for this experiment. This demonstrates that deployment performance can be further optimized by adjusting the decision threshold according to business requirements.
However, for simplicity this assignment only considers the default threshold.

Result comparision between baseline scaled, tuned at threshold 0.5, tuned at threshold 0.8

<pre><b>
Model               Layers  Neurons     Activation      Threshold       Learning Rate   Precision   Recall      F1          PR-AUC</b>
Baseline scaled     2       32, 16      relu, relu      0.5 (default)   0.001 (default) 0.404110    0.830986    0.543779    0.754427
Fine tuned          1       64, 00      relu, leakyrelu 0.5             0.001           0.442748    0.816901    0.574257    0.719718
Fine tuned w/       1       64, 00      relu, leakyrelu 0.8             0.001           0.585859    0.816901    0.682353    0.719718
Threshold
<!-- Model               Layers  Neurons     Activation      Threshold       Learning Rate   Precision   Recall      F1          PR-AUC</b>
Baseline scaled     2       32, 16      relu, relu      0.5 (default)   0.001 (default) 0.537037    0.816901    0.648045    0.781371
Fine tuned          2       64, 32      relu, relu      0.5             0.001           0.578431    0.830986    0.682081    0.788707
Fine tuned w/       2       64, 32      relu, relu      0.8             0.001           0.698795    0.816901    0.753247    0.788707
Threshold -->
</pre>

Optimizer for all experiments: Adam
</div>

In [ ]:
# track the total time for section
xgb_tune_start = time.time()
xgb_tune_time = record_time(None, START, xgb_tune_start)
timings[XGB_TUNING] = xgb_tune_time

#### Tune XGBoost

hyperparams = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [3, 5, 7, 9],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "min_child_weight": [1, 3, 5],
    "gamma": [0, 0.1, 0.3, 0.5]
}

model = xgb.XGBClassifier(
    # objective="binary:logistic", # this is default anyway
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
    scale_pos_weight=scale_pos_weight
)

tuner = build_model_tuner(model, hyperparams, nIterations=30)
tuner.fit(X_train, y_train) # tune the model

# track the total time for section
xgb_tune_time = record_time(xgb_tune_time, END, time.time())

# track the total time for section
sec_time = record_time(None, START, time.time())
timings[f"Retrained Tuned {XG_BOOST}"] = sec_time

model_name = f"{XG_BOOST}_tuned"
best_hp = tuner.best_params_
print(f"Best score {tuner.best_score_} with following best params for model {model_name}: ", best_hp)
print(f"Cross validation results:")
prettyPrint(pd.DataFrame(tuner.cv_results_).sort_values(
    "rank_test_score"
).head())

model = xgb.XGBClassifier(
    **best_hp, # expand all the best params that tuner provided
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
    scale_pos_weight=scale_pos_weight
)

metrics, unified_metrics = run_experiment(model_registry, model_name, model_name, model, None, X_train, y_train, X_test, y_test, X_train_scaled, X_test_scaled, unified_metrics = unified_metrics, add_to_unified_metrics=True, plot_auc=True, scaled_data=False, scaler=scaler)

# track the total time for section
sec_time = record_time(sec_time, END, time.time())

In [ ]:
# track the total time for section
rf_tune_time = record_time(None, START, time.time())
timings[RF_TUNING] = sec_time

#### Tune RandomForest

hyperparams = {
    "n_estimators": [100, 200, 300],
    "max_depth": [10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt"],
    "bootstrap": [True]
}

nIterations = 10 # changed from 15 to 2 for testing the excution environment where the joblib, tuner, threads etc interact
tuner_nJobs = 1 # -1 or 4 don't return at all even after hours so turning off parallelism to see if that helps
estimator_nJobs = -1
randState = 42

"""With tuner_njobs > 1, there is a lot of parallelism that it tries which probably causes deadlocks or simply tremendous slowdown because
of which even the first CV never completes (no CV 1/5 END output). estimator_njobs creates parallel trees for that one forest so that works
when only the estimator is running alone without the tuner.
When both nJobs > 1, it triggers parallelism leading to weird interaction between windows, loky, VS Code notebook, and joblib multiprocessing.
This causes things to become dead slow. When tuner_njobs = 1, joblib multiprocessing is not triggered causing things to move.
When tuner_njobs=1 and estimator_njobs=-1, tuner's cross validation happens sequentially but the random forest itself creates multiple
trees in parallel improving performance signifcantly. On this machine at this experiment improvement in speed was >3x - with both njobs=1
total time was about 24 mins, with estimator_njobs=-1, it was about 7 mins for 2 iterations!
"""

model = en.RandomForestClassifier(
    n_jobs=estimator_nJobs,  #instead of -1 here use 1 to use only 1 cpu core because the tuner is using all cores (-1) which might slow down the machine and thus entire processing
    random_state=randState,
    class_weight="balanced"
)

print("Sampled params that the tuner will use:")
samples = list(ParameterSampler(
    hyperparams,
    n_iter=nIterations,
    random_state=randState
))

for i, params in enumerate(samples, 1):
    print(f"Candidate {i}: {params}")

print("Building tuner")
tuner = build_model_tuner(model, hyperparams, nIterations = nIterations, nJobs=tuner_nJobs, randSatate=randState)
print("completed building tuner")

print("starting tuning")
tuner.fit(X_train, y_train) # tune the model
print("completed tuning")

# track the total time for section
rf_tune_time = record_time(rf_tune_time, END, time.time())

# track the total time for section
sec_time = record_time(None, START, time.time())
timings[f"Retraining Tuned {RANDOM_FOREST}"] = sec_time

model_name = f"{RANDOM_FOREST}_tuned_reduced_search_space_noParallelTuner_onlyParallelEstimator"
best_hp = tuner.best_params_
print(f"Best score {tuner.best_score_} with following best params for model {model_name}: ", best_hp)
print(f"Cross validation results:")
prettyPrint(pd.DataFrame(tuner.cv_results_).sort_values(
    "rank_test_score"
).head())

model = en.RandomForestClassifier(
    **best_hp, # expand all the best params that tuner provided
    n_jobs=estimator_nJobs,
    random_state=randState,
    class_weight="balanced"
)

metrics, unified_metrics = run_experiment(model_registry, model_name, model_name, model, None, X_train, y_train, X_test, y_test, X_train_scaled, X_test_scaled, unified_metrics = unified_metrics, add_to_unified_metrics=True, plot_auc=True, scaled_data=False, scaler=scaler)

end_time = time.time()
# track the total time for section
sec_time = record_time(sec_time, END, end_time)

# track the total time for section
experiments_sec_time = record_time(experiments_sec_time, END, end_time)

In [ ]:
# Find feature importances by gain

from xgboost import plot_importance

xgb_tuned = model_registry[f"{XG_BOOST}_tuned"]["model"]

# Feature importance (Gain)
importance_gain = pd.DataFrame({
    "Feature": X_train.columns,
    "Gain": xgb_tuned.feature_importances_
}).sort_values("Gain", ascending=False)

print(xgb_tuned.feature_importances_)
prettyPrint(importance_gain)


# Plot top 20
plt.figure(figsize=(8,6))
plot_importance(
    xgb_tuned,
    importance_type="gain",
    max_num_features=20,
    height=0.5
)
plt.title("XGBoost Feature Importance (Gain)")
plt.tight_layout()
plt.show()

In [ ]:
# feature importance by permutaion importance
from sklearn.inspection import permutation_importance

perm_importance = permutation_importance(
    xgb_tuned,
    X_test,
    y_test,
    scoring="average_precision",   # PR-AUC
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

perm_df = pd.DataFrame({
    "Feature": X_test.columns,
    "Importance": perm_importance.importances_mean,
    "Std": perm_importance.importances_std
}).sort_values("Importance", ascending=False)

prettyPrint(perm_df)

<b>Final observations</b>
<ul>
    <li>
        Raw vs scaled data
        <div>
            <ol>
                <li>
Tree-based algorithms (Decision Tree, Random Forest, XGBoost, and Gradient Boosting) are largely insensitive to feature scaling. Their raw and scaled results remained very similar, with only minor variations likely due to differences in split selection caused by numerical precision.
</li>
                <li>
KNN benefited substantially from feature scaling. Test precision improved from 0 to 0.9592, recall from 0 to 0.6620, F1-score from 0 to 0.7833, and PR-AUC from 0.0414 to 0.7790.
</li>
            </ol>
        </div>
    </li>
    <li>
<div style="padding-top:5px">
Top baseline models (default threshold = 0.5) ranked primarily by <b>PR-AUC</b>:
</div>

<ul>
<li>
XGBoost (Raw): PR-AUC = 0.812129, Recall = 0.802817, F1 = 0.786207, Precision = 0.770270
</li>

<li>
Random Forest (Raw): PR-AUC = 0.802100, Recall = 0.690141, F1 = 0.803279, Precision = 0.960784
</li>

<li>
KNN (Scaled): PR-AUC = 0.778996, Recall = 0.661972, F1 = 0.783333, Precision = 0.959184
</li>

<li>
Best Deep Learning Model (ANN Scaled): PR-AUC = 0.754427, Recall = 0.830986, F1 = 0.543779, Precision = 0.404110
</li>
</ul>
</li>
    <li>
        <div>
            Generalization of top base line models (all with default threshold 0.5) based on training <b>PR-AUC, recall, F1, and precision</b>:
            <ul>
                <li>
                    XGBoost on raw data (0.994816, 1.000000, 0.937063, 0.881579)
                </li>
                <li>
                    ANN on scaled data (0.937922, 0.977612, 0.685266, 0.527517)
                </li>
                <li>
                    Random Forest on raw data (1.000000, 1.000000, 1.000000, 1.000000)
                </li>
            </ul>
            <div style="padding-bottom:5px">
                While all three baseline models learned the training data extremely well, each showed some reduction in performance on the unseen test set. This indicates a degree of overfitting, although their test performance remained strong relative to the other baseline models.
            </div>
        </div>
    </li>
    <li>
        Fine tuned top models (all with default threshold 0.5) based on <b>PR-AUC, recall, F1, and precision</b>:
        <div>
            <ul>
                <li>
                    XGBoost on raw data (0.815841, 0.802817, 0.863636, 0.934426)
                    Fine tuned params: {'subsample': 0.7, 'n_estimators': 500, 'min_child_weight': 1, 'max_depth': 5, 'learning_rate': 0.05, 'gamma': 0, 'colsample_bytree': 1.0}
                </li>
                <li>
                    ANN on scaled data (0.719718, 0.816901, 0.574257, 0.442748)
                    <div>
                        Fine tuned params: {
'num_layers':1,
'units_1':64,
'activation_1':'relu',
'learning_rate':0.001
}
                        <div style="padding-top:5px">
                        <b>Note</b>: Threshold tuning analysis was done and with increasing threshold results consistently improved but for consistent comparision with other models where threshold tuning was not done, results at threshold of 0.5 were considered. 
                        <span style="display:block; padding-top:10px; padding-bottom:10px">Results at highest threshold of 0.8 were (0.719718,	0.816901, 0.682353, 0.585859).</span>
                        This demonstrates that deployment performance can be further optimized by adjusting the decision threshold according to business requirements.
                        </div> 
                    </div>
                </li>
                <li>
                    Random Forest on raw data (0.810883, 0.718310, 0.816000, 0.944444)
                    Fine tuned params: {'n_estimators': 200, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': 20, 'bootstrap': True}
                </li>
            </ul>
        </div>
    </li>
    <li>
        <b>Feature Importance</b>
<div>
<b>Feature importance by gain</b> was evaluated using the final tuned XGBoost model based on the Gain metric, which measures the average improvement in the model's objective function contributed by each feature when it is used for splitting.

The model identified <b>V14</b> as the strongest predictor of fraudulent transactions with a gain importance of <b>0.478</b>, substantially higher than any other feature. The next most influential features were <b>V4</b> (0.067), <b>V20</b> (0.041), <b>V12</b> (0.035), and <b>V10</b> (0.029). Together, these features contributed the majority of the model's predictive power.

Among the non-PCA variables, <b>AMOUNT</b> had modest importance (0.014), while <b>TIME</b> contributed relatively little (0.012). This suggests that the transformed PCA components capture most of the information required to distinguish fraudulent from legitimate transactions.
</div>
<div>
Top 10 features by gain importance include:
<pre>
    <b>Rank    Feature Gain    Importance</b>
    1       V14             0.478
    2       V4              0.067
    3       V20             0.041
    4       V12             0.035
    5       V10             0.029
    6       V19             0.026
    7       V8              0.024
    8       V11             0.023
    9       V3              0.019
    10      V7              0.018
</pre>
</div>
<div>
    <b>Feature importance by permutaton</b> was computed using PR-AUC on the test set to estimate each feature's contribution to the final model. The results confirmed that V14, V4, and V12 were the most influential predictors of fraudulent transactions, consistent with the Gain-based feature importance. Several features, including TIME, V9, V13, V16, V22, and V24, had near-zero or slightly negative permutation importance, indicating that they contributed little unique predictive information to the final model. The agreement between Gain-based and Permutation importance for the top-ranked features increases confidence that these variables drive most of the model's predictive performance.
</div>
<div>
Top 10 features by permutation importance include:
<pre>
    <b>Rank    Feature Importance  Std</b>
    1       V14     0.126504    0.019054
    2       V4      0.080464    0.015460
    3       V12     0.041183    0.019448
    4       V11     0.017959    0.013392
    5       V3      0.017878    0.013688
    6       V28     0.012092    0.003727
    7       V26     0.011649    0.003765
    8       V8      0.008884    0.003377
    9       V10     0.006334    0.008895
    10      V27     0.005868    0.002251
</pre>
</div>
    </li>
</ul>

<b>Final Model Selection for deployment: Tuned XGBoost</b><br/>
Models were primarily compared using PR-AUC, as it is the most appropriate metric for highly imbalanced binary classification problems. Recall, precision, and F1-score were then examined to assess the trade-off between fraud detection capability and false alarms. ROC-AUC was used as a secondary measure of ranking performance, while accuracy was reported for completeness but not used for model selection.

Based on the baseline results, XGBoost and Random Forest achieved the highest PR-AUC among the classical machine learning models and were therefore selected for hyperparameter tuning. Although the scaled ANN was not among the top three models by PR-AUC, it was also selected for tuning as the best-performing deep learning model to enable a fair comparison between optimized machine learning and neural network approaches.

<b>Tuned XGBoost</b> was selected as the final deployment model because it achieved the best overall balance of precision, recall, F1-score, ROC-AUC, and PR-AUC on the unseen test set. It consistently outperformed the tuned Random Forest and all ANN variants while maintaining excellent generalization and achieving the highest overall balance between fraud detection and false positives without needing feature scaling.
<pre>
<b>
Model           Data    Accuracy    Precision   Recall      F1          ROC_AUC     PR_AUC</b>
XGBoost         Raw     0.999577    0.934426    0.802817    0.863636    0.969731    0.815841
Random Forest   Raw     0.999460    0.944444    0.718310    0.816000    0.932629    0.810883
ANN             Scaled  0.997979    0.442748    0.816901    0.574257    0.959525    0.719718
</pre>

In [ ]:
# track the total time for section
sec_time = record_time(None, START, time.time())
timings[MODEL_DEPLOYMENT] = sec_time

def build_feature_metadata(datadf):
    """
    Build feature metadata directly from the training dataframe.

    Returns
    -------
    dict
        {
            "all": {
                    feature_name: {
                    "dtype": ...,
                    "min": ...,
                    "max": ...,
                    "mean": ...,
                    "median": ...,
                    "std": ...,
                    "q1": ...,
                    "q3": ...,
                    "iqr": ...
                }
            },
            "non-fraud": {
                    feature_name: {
                    "dtype": ...,
                    "min": ...,
                    "max": ...,
                    "mean": ...,
                    "median": ...,
                    "std": ...,
                    "q1": ...,
                    "q3": ...,
                    "iqr": ...
                }
            },
            "fraud": {
                    feature_name: {
                    "dtype": ...,
                    "min": ...,
                    "max": ...,
                    "mean": ...,
                    "median": ...,
                    "std": ...,
                    "q1": ...,
                    "q3": ...,
                    "iqr": ...
                }
            }
            
        }
    """

    metadata = {"all": {}, "non-fraud": {}, "fraud": {}}

    # Get basic stats like mean, median, quartiles etc for entire dataset and for each class. This will be used to generate random samples
    # during inference to assist the user in filling the large form. To demonstrate that the model actually detects frauds, samples from
    # both categories will be generated and the model should be able to recall most frauds.

    # For entire dataset, the stats were generated as part of EDA, use the same desc dataset for that. For classes, use below sets
    # get stats for non-fraud class
    desc_normal = describeData(df.loc[df["CLASS"] == 0], excludeCols=categoricalColumns)

    # get stats for fraud class
    desc_fraud = describeData(df.loc[df["CLASS"] == 1], excludeCols=categoricalColumns)

    for s in datadf.dtypes.index:
        if(s.lower() == "class"):
            continue
        
        dataType = datadf.dtypes[s]
        metadata["all"][s] = {
            "dtype": str(dataType),
            "min": float(desc[s]["min"]),
            "max": float(desc[s]["max"]),
            "mean": float(desc[s]["mean"]),
            "median": float(desc[s]["50%"]),
            "std": float(desc[s]["std"]),
            "q1": float(desc[s]["25%"]),
            "q3": float(desc[s]["75%"]),
            "iqr": float(desc[s]["75%"] - desc[s]["25%"])
        }

        metadata["non-fraud"][s] = {
            "dtype": str(dataType),
            "min": float(desc_normal[s]["min"]),
            "max": float(desc_normal[s]["max"]),
            "mean": float(desc_normal[s]["mean"]),
            "median": float(desc_normal[s]["50%"]),
            "std": float(desc_normal[s]["std"]),
            "q1": float(desc_normal[s]["25%"]),
            "q3": float(desc_normal[s]["75%"]),
            "iqr": float(desc_normal[s]["75%"] - desc_normal[s]["25%"])
        }

        metadata["fraud"][s] = {
            "dtype": str(dataType),
            "min": float(desc_fraud[s]["min"]),
            "max": float(desc_fraud[s]["max"]),
            "mean": float(desc_fraud[s]["mean"]),
            "median": float(desc_fraud[s]["50%"]),
            "std": float(desc_fraud[s]["std"]),
            "q1": float(desc_fraud[s]["25%"]),
            "q3": float(desc_fraud[s]["75%"]),
            "iqr": float(desc_fraud[s]["75%"] - desc_fraud[s]["25%"])
        }

    return metadata

# create the deployment pipeline for the selected model; this will be used to deploy the model in production and will include all the 
# necessary steps such as scaling, encoding, and model training

final_model_name = f"{XG_BOOST} Scaled" # simply change the model name here copying it from the train-test results printed above
selected_model = model_registry[final_model_name]

deployment_pipeline = Pipeline([
    (f"step_{i}", step)
    for i, step in enumerate(selected_model["steps"])
])

# create the deployment artifact
deployment_artifact = {
    "model_name": final_model_name,
    "pipeline": deployment_pipeline,
    "features": list(selected_model["features"]),
    "feature_metadata": build_feature_metadata(df),
    "metrics": selected_model["metrics"],
    "imbalance": selected_model["imbalance"],
    "data_type": selected_model["data"]
}

# save the artifact
joblib.dump(
    deployment_artifact,
    "credit_card_fraud_detector.joblib",
    compress=3
)

end_time = time.time()
# track the total time for section
sec_time = record_time(sec_time, END, end_time)

# record end time of entire program
sec_time = record_time(prog_time, END, end_time)

In [ ]:
# print the timings
print("Time taken by various sections:")
for sec in timings:
    print(f"{sec+":":<20} {(timings[sec][END] - timings[sec][START]):7.2f} secs (Start: {format_time(timings[sec][START])} End: {format_time(timings[sec][END])})")

In [ ]:
# Load the artifact, retrieve the pipeline
artifact = joblib.load("credit_card_fraud_detector.joblib")
pipeline = artifact["pipeline"]
features = artifact["features"]

#### Sanity test things out with original dataset; for testing it via UI, use the streamlit version where random samples are generated 
# instead of using them from the training set

# Number of normal and fraud samples to test
num_normal = 5
num_fraud = 5

# -----------------------------
# Build a small test dataset
# -----------------------------

# First few normal transactions (head of dataset)
normal_samples = X_test[y_test == 0].sample(n=5)[features]

# First few fraud transactions
fraud_samples = X_test[y_test == 1].sample(n=5)[features]

# Combine them
X_demo = pd.concat(
    [normal_samples, fraud_samples],
    axis=0
)

# Corresponding true labels
y_demo = pd.concat(
    [y_test.loc[normal_samples.index],
     y_test.loc[fraud_samples.index]],
    axis=0
)

# -----------------------------
# Run inference
# -----------------------------

predictions = pipeline.predict(X_demo)
probabilities = pipeline.predict_proba(X_demo)[:, 1]

# -----------------------------
# Display results
# -----------------------------

results = X_demo.copy()

results["Actual"] = y_demo.values
results["Predicted"] = predictions
results["Fraud Probability"] = probabilities

display(results)

In [ ]:
####### Github link: https://github.com/Atul-Lohiya/Capstone02CreditCardFraudDetection


####### Streamlit code: Select entire code and uncomment by pressing CTRL + /

# # """
# # Section 1 - Imports, artifact loading, metadata extraction and helper utilities.
# # Append Sections 2 and 3 below this in the same app.py.
# # """

# import json
# import random
# from pathlib import Path

# import joblib
# import numpy as np
# import pandas as pd
# import streamlit as st

# st.set_page_config(page_title="Credit Card Fraud Detection", layout="wide")

# ARTIFACT_PATH = "credit_card_fraud_detector.joblib"

# @st.cache_resource
# def load_artifact():
#     artifact = joblib.load(ARTIFACT_PATH)
#     return artifact

# artifact = load_artifact()
# pipeline = artifact["pipeline"]
# feature_order = artifact["features"]
# metadata = artifact.get("feature_metadata", {}) # this has stats like mean, median, quratiles etc for all, non-fraud and fraud classes
# metadata_all = metadata["all"]
# metadata_non_fraud = metadata["non-fraud"]
# metadata_fraud = metadata["fraud"]
# # feature_order = metadata.keys()

# def feature_info(feature, sample_class):
#     meta = metadata_all

#     if sample_class == 0:
#         meta = metadata_non_fraud
#     elif sample_class == 1:
#         meta = metadata_fraud

#     # return metadata.get(feature, {})
#     return meta.get(feature, {})

# def random_value(feature, sample_class):
    
#     info = feature_info(feature, sample_class)
#     median = info.get("median")
#     mn = info.get("min")
#     mx = info.get("max")
#     iqr = info.get("iqr")
#     sigma = iqr / 1.349 # standard formula for IQR = 1.349 * sigma
#     value = float(np.random.normal(median, sigma))
#     # return np.clip(
#     #     value,
#     #     mn,
#     #     mx)   # clip if you need to constrain it between min and max for the training dataset, if not return the unbounded value
#     return value

# def random_record():
#     choice = random.choices(
#                 population=[-1, 0, 1],
#                 weights=[0.375, 0.375, 0.25],
#                 k=1
#             )[0]
#     record = {}
#     for f in feature_order:
#         value = random_value(f, choice)
#         if f.lower() == "amount" or f.lower() == "time":
#             value = max(0.0, value)
#         record[f] = value
#         record["class"] = choice
#     return record

# def records_to_dataframe(records):
#     df = pd.DataFrame(records)
#     return df[feature_order]

# def predict_records(records):
#     df = records_to_dataframe(records)
#     pred = pipeline.predict(df)
#     prob = pipeline.predict_proba(df)[:,1]
#     out = df.copy()
#     out["Prediction"] = pred
#     out["Fraud Probability"] = prob
#     return out


# # """
# # Section 2 - Streamlit UI.
# # Append below Section 1.
# # """

# st.title("Credit Card Fraud Detection")

# # =============================================================================
# # Information about Synthetic Data Generation
# # =============================================================================

# with st.expander("About Synthetic Sample Generation", expanded=False):

#     st.markdown("""
# ### Why synthetic sample generation?

# This application provides automatic sample generation to make it easier to
# demonstrate model inference.

# The dataset contains **28 PCA-transformed features (V1–V28)** whose original
# business meanings are intentionally unavailable. Because these principal
# components do not have directly interpretable ranges, manually creating
# realistic transactions can be difficult.

# Synthetic transaction generation helps users quickly test the deployed model.
# The generated values may also be manually edited before prediction.

# ---

# ### How are random transactions generated?

# Three statistical profiles are derived from the training dataset:

# - **Overall dataset (class=-1)**
# - **Non-Fraud transactions (class=0)**
# - **Fraud transactions (class=1)**

# Whenever a transaction is generated, one of these profiles is randomly selected
# using weighted sampling.

# Each feature is then generated independently by sampling from a **Normal
# distribution** centered on the feature's **median**, with the spread estimated
# from the feature's **Interquartile Range (IQR)**.

# For the **TIME** and **AMOUNT** features, negative values are prevented since
# they are not meaningful.

# ---

# ### Important note

# The generated transactions are **synthetic** and are intended only for
# demonstrating model inference.

# Feature values are generated independently and therefore **do not preserve the
# true relationships (correlations)** that exist among variables in real credit
# card transactions. Consequently, these samples should **not** be interpreted as
# real transactions or used for model evaluation.

# Model performance reported in this project is based exclusively on the original
# training and test datasets.
# """)

#     st.subheader("Training Data Statistics")

#     tab1, tab2, tab3 = st.tabs([
#         "Overall (class=-1)",
#         "Non-Fraud (class=0)",
#         "Fraud (class=1)"
#     ])

#     with tab1:
#         st.dataframe(
#             pd.DataFrame(metadata_all).T,
#             use_container_width=True
#         )

#     with tab2:
#         st.dataframe(
#             pd.DataFrame(metadata_non_fraud).T,
#             use_container_width=True
#         )

#     with tab3:
#         st.dataframe(
#             pd.DataFrame(metadata_fraud).T,
#             use_container_width=True
#         )

# st.info(
# """TIME is seconds since the first transaction in the dataset.
# V1-V28 are PCA components. You can either fill them manually or click "Auto-fill fields" to aut-fill them based on deployed training metadata like quartiles, min, max etc.
# Deployment uses the default classification threshold of 0.5, matching model evaluation."""
# )

# num_cols = 5
# cols=st.columns(num_cols)
# class_mapping = {
#     -1: "Overall Training Statistics (class=-1)",
#      0: "Non-Fraud Statistics (class=0)",
#      1: "Fraud Statistics (class=1)"
# }

# for i,f in enumerate(feature_order):
#     col=cols[i % num_cols]
#     # info=feature_info(f, -1)
#     with col:
#         st.number_input(
#             f,
#             # value=float(info.get("median",0.0)),  # set median as default
#             value=float(0.0),
#             # min_value=float(info.get("min",-1e9)),
#             # max_value=float(info.get("max",1e9)),
#             step=0.01,
#             key=f
#         )
# if "class" in st.session_state:
#     st.info(
#         f"Random sample generated using **{st.session_state['class']}**."
#     )

# def autofill_fields():
#     record = random_record()

#     for k, v in record.items():
#         # if k == "class": # skip class property as that is not a user field
#         #     continue
#         st.session_state[k] = class_mapping[v] if k == "class" else v

# buttons_row = st.columns([1, 1, 10], gap="small")
# with buttons_row[0]:
#     autofill = st.button("Auto-fill", on_click=autofill_fields)

# with buttons_row[1]:
#     predict_single = st.button("Predict")

# with buttons_row[2]:
#     st.html("""
#     <div style="
#         display:flex;
#         align-items:center;
#         height:38px;
#         font-size:14px;
#         color:#666;
#     ">Scroll to bottom for results</div>
#     """)

# st.divider()
# st.subheader("Batch prediction (JSON)")
# st.write("Auto-generate records")
# buttons_row = st.columns([1.5, 1.5, 9], gap="small")

# with buttons_row[0]:
#     count=st.number_input("Auto-generate records",1,100,50, label_visibility="collapsed")
# with buttons_row[1]:
#     gen_json=st.button("Generate JSON")
# with buttons_row[2]:
#     predict_json=st.button("Predict JSON")

# if gen_json:
#     st.session_state["json_text"]=json.dumps(
#         [random_record() for _ in range(int(count))],
#         indent=2
#     )

# json_text=st.text_area(
#     "JSON array of transaction objects",
#     value=st.session_state.get("json_text",""),
#     height=350
# )



# # """
# # Section 3 - Inference, Results and Main App

# # Paste below Sections 1 and 2.
# # """

# # ---- Individual prediction ----
# if predict_single:
#     rec = {k: st.session_state[k] for k in feature_order}
#     # rec = st.session_state["single_record"]
#     df = records_to_dataframe([rec])
#     probs = pipeline.predict_proba(df)[:,1]
#     preds = pipeline.predict(df)
#     out = df.copy()
#     out["Prediction"]=preds
#     out["Fraud Probability"]=probs
#     st.dataframe(out, use_container_width=True)

# # ---- Batch prediction ----
# if predict_json:
#     try:
#         records=json.loads(json_text)
#         if not isinstance(records,list):
#             raise ValueError("JSON must be a list of objects.")
#         df=records_to_dataframe(records)
#         probs=pipeline.predict_proba(df)[:,1]
#         preds=pipeline.predict(df)
#         out=df.copy()
#         out["Prediction"]=preds
#         out["Fraud Probability"]=probs
#         st.dataframe(out,use_container_width=True)
#     except Exception as e:
#         st.error(str(e))



#### Scratchpad for troubleshooting

In [ ]:
prettyPrint(unified_metrics[
    (unified_metrics["model"].str.lower().str.contains("scaled-fine-tuned", na=False)) &
    (unified_metrics["eval_type"] == "Test")
    ])

In [ ]:
prettyPrint(unified_metrics[
    (unified_metrics["model"].str.lower().str.contains("xgboost", na=False))
    #  &
    # (unified_metrics["eval_type"] == "Test")
    ])

In [ ]:
prettyPrint(unified_metrics[
    (unified_metrics["model"].str.lower().str.contains("forest", na=False))
    #  &
    # (unified_metrics["eval_type"] == "Test")
    ])

In [ ]:
model_registry